# LLM-Powered Stock Trading Agent

## Paper Trading with LLM Decision Making, News Sentiment & RAG Memory

---

**Student Information**

Author: Syeda Tamzida Akter


> **Update (Oct 11):** Alpha Vantage made news (`NEWS_SENTIMENT`) and `MACD` premium-only. Section 1.9 now gets news from **Alpaca**, and Task 2 computes MACD from **yfinance** prices. RSI and GLOBAL_QUOTE still use Alpha Vantage.

---

### Assignment Overview

In this assignment, you will build components of an **AI-powered stock trading agent** that:
1. Uses real-time financial news and sentiment data to inform decisions
2. Extends data sources with custom retrieval functions (MCP Styled Tools)
3. Learns from past trades using a RAG memory system (BM25 + Reflection)
4. Executes paper trades via the Alpaca brokerage API

### Task Breakdown

| Task | Topic | Status |
|------|-------|--------|
| **Task 1** | Environment, Infrastructure & Core Logic | TODO |
| **Task 2** | MCP Tool Functions (Additional Data Sources) | TODO |
| **Task 3** | RAG Memory System (BM25 + Reflection) | TODO |
| **Task 4** | SFT Fine-tuning with LoRA | TODO |
| **Task 5** | Full Trading Loop Integration | TODO |


### Running locally (Windows + Jupyter) instead of Colab

Changes from the original framework, made only so the notebook runs on a local machine (NVIDIA RTX 5070 Laptop GPU, 8 GB):

| Cell | Original | Changed to | Why |
|---|---|---|---|
| 1.1 Install Dependencies | `!pip install ...` | commented out | packages installed once in the `trading` conda environment |
| 1.3 Mount Google Drive | `drive.mount(...)`, `CACHE_DIR="/content/drive/MyDrive/stock_agent_cache"` | `CACHE_DIR = ./stock_agent_cache` | no Google Drive locally |
| 1.4 Configuration & API Keys | keys pasted in or read from Colab Secrets | keys read from a local `.env` file | Colab Secrets unavailable; keeps keys out of the notebook |
| Inspect & Edit cells (after 4a and Task 5) | `from google.colab import files` | removed; files opened with `encoding="utf-8"` | Colab-only module; Windows default encoding is not UTF-8 |

Everything the notebook describes as saved "on Drive" (news cache, `sft_data.jsonl`, `lora_adapter`, `trading_logs`) is saved in `stock_agent_cache` next to this notebook.

### Changes to the framework (Tasks 1.9, 1.10, 2)

| Cell | Change | Why |
|---|---|---|
| 1.9 `get_news_for_symbols` | Cache stores each ticker batch separately inside the date file (`batches`), plus the `text/timestamp/date/symbols` keys the test expects | the original kept one text per date, so a later call with different tickers on the same date returned the wrong news |
| 1.9 | Backtest window ends at `20:00Z` instead of `23:59:59Z` | news published after the market close must not influence that day's decision (look-ahead) |
| 1.9 | Live cache refreshes a batch after 30 minutes | otherwise the agent sees the morning's headlines all day |
| 1.9 | Each article gets an evidence ID `[N-<alpaca id>]` | lets the agent cite news; 1.12 verifies cited IDs |
| 1.10 `dispatch_tool` | Passes the backtest `date` to every tool that has a `date` parameter and drops any `date` the model supplies; accepts `"AAPL,NVDA"` strings for `get_news` | originally only `get_news` got the date, so indicators in the backtest used today's data (look-ahead) |
| 1.10 `get_news` description | removed "+ sentiment" | Alpaca news has no sentiment scores |
| Task 2 | All tools take an optional `date`; data after that date is cut off | point-in-time backtest |
| Task 2 `get_rsi` | Alpha Vantage RSI series cached per symbol (one call covers all dates); RSI computed from yfinance if Alpha Vantage fails | 25 calls/day free limit |
| Task 2 `get_global_quote` | Alpha Vantage quote cached 15 min, yfinance fallback; in the backtest returns that day's close | call limit; a live quote for a past day would leak today's price |
| Task 2 | Daily call budget for Alpha Vantage (`AV_DAILY_BUDGET = 22`) | never exceed the free tier |
| Task 2 new tools | `get_company_research` (SEC EDGAR 8-K/10-Q/10-K + quarterly EPS/revenue YoY, only filings before the decision date), `get_market_context` (SPY trend + VIX), `research_flags` (large move or earnings just released) | the evidence-grounded research component of my proposal |
| Task 2 | Every fact carries an evidence ID (`[P-]`, `[R-]`, `[D-]`, `[Q-]`, `[S-]`, `[F-]`, `[M-]`) | the agent must cite evidence; unsupported trades are blocked in 1.12 |

### Changes to the framework (1.4, 1.12–1.14, Tasks 3–5)

| Cell | Change | Why |
|---|---|---|
| 1.4 | `LOOP_SECONDS` 10 → 120, `REFLECTION_DELAY_LOOPS` 30 → 20; new settings `FOCUS_LIST`, `USE_RESEARCH`, `USE_VERIFICATION`, `MAX_TOOL_CALLS`, `MAX_POSITION_PCT`, `DAILY_LOSS_LIMIT`, `TRANSACTION_COST_BPS`, `RUN_MINUTES` | realistic live interval (80 loops ≈ 4 h); a reflection after ≈ 1 h; one place for all agent settings |
| 1.12 | Qwen chat template (`apply_chat_template` with tool schemas, `tool` messages) instead of the hand-written `<|system|>` format | the 1.7 test showed the old format makes the model ramble ("Human: …"); Qwen is trained on its own template |
| 1.12 | Agent sees a 10-stock focus list (+ held stocks) with a market snapshot; stocks with a large move / fresh earnings are marked RESEARCH NEEDED | a 1.5B model cannot sensibly scan 100 tickers; research only when it is worth it |
| 1.12 | Decisions must cite evidence IDs; Python verifies each buy/sell cites an ID that appeared in this conversation and is about that stock, otherwise it becomes HOLD | proposal rule: a claim with no source cannot be the reason for a trade |
| 1.12 | Risk checks: max qty, max 15% of equity per stock, cash limit, no short selling, half size in RISK-OFF, no buys after the daily loss limit | risk manager |
| 1.12 | One correction round when no trade passes the checks; at least one tool call required; max 4 tool calls | robust behaviour for a small model |
| 1.12 | Fixed random seed per backtest day | base vs fine-tuned compared on the same random draws |
| 1.13 | New `label_by_forward_return` (3-day forward return, 0.5% no-trade band; good / corrected / bad / neutral) | proposal's labelling rule; next-day portfolio P&L mixes all positions and luck |
| 1.14 | `run_backtest(skip_last=…)`; logs each day's date; 0.05% transaction cost; buy-and-hold SPY and equal-weight benchmarks; prices downloaded for 6 months | held-out test period with no overlap; fair comparison |
| 3a | BM25 with regex tokens + word-overlap backup score | with only 1–2 memories BM25's IDF is ≤ 0 and nothing would be retrieved |
| 3b | Reflection starts with a fixed "ACTION QTY SYMBOL: gained/lost $X" part, then the model's one-sentence lesson | every reflection names stock, action and outcome |
| 4a | Backtest the 40 days before the last 20; label with my rule | the last 20 days are the held-out test for 4c |
| 4b | Completion ends with `<|im_end|>`; prompt truncated from the left; no padding; gradient checkpointing; skip non-finite losses; gradient clipping | completion never cut off (NaN loss); fits in 8 GB |
| 4c | Training count includes corrected days; prints benchmarks and agent stats; new improvement cell (research + verification off vs on) | report evidence |
| Task 5 | Full loop: market clock, loss limit, focus symbols (positions + 3 biggest movers), news, Task 2 data, reflections, memory retrieval, decision, orders only for buy/sell while the market is open | Task 5 |
| Final cell | `max_minutes=RUN_MINUTES` | stop after ~4 hours |

### Fix after the first training run (v3 → v4) — a second before/after

| | First run (v3) | Fix (v4) |
|---|---|---|
| Problem seen in 4c | Fine-tuned model wrote tool calls as plain JSON (`[{"name": "get_news", ...}]`) without `<tool_call>` tags, so **0 tool calls on all 20 days**; it then invented evidence IDs (`M-META_1mo`, `N-12345`, `RESEARCH_NEEDED`) and **all 20 trades were blocked** → 0 trades, return +0.00% only because it held cash | |
| Cause | SFT labelled only the final decision JSON; 15 examples × 3 epochs (loss 0.20) overfitted, so the model answered every turn with a JSON array | |
| 1.12 `_parse_tool_calls` | only `<tool_call>` blocks | also accepts plain-JSON tool calls whose `name` is a registered tool |
| 4b `prepare_sft_dataset` | loss on the final decision only | loss also on the earlier **tool-call turns** of each good day (turns that needed a correction stay masked) |
| 4b `train_sft` | epochs 3, lr 2e-4 | epochs 2, lr 1e-4 (less overfitting) |
| Evidence | v3 notebook, 4c output | v4 notebook, 4c output |


---
## Task 1: Environment, Infrastructure & Core Logic

### Grading criteria

- [ ] `llm_agent_decide()` (1.12) runs the multi-turn tool loop and returns 4 values. *Evidence: 1.15 output (rerun after Task 2) shows, turn by turn, the tool called → the first lines of its result → the model's final raw text → a decision that can be executed.*
- [ ] Most days in the 4a backtest have a real decision: the model calls a tool and gives at least one valid buy or sell, rather than the code falling back to a default. *Evidence: 4a output: `Episodes` (days logged) out of the backtest days. Run 4a before 4c, and keep the turn-by-turn log.*
- If your agent can HOLD, explain its HOLD decisions in your report with 2–3 examples from your log.


### 1.1 Install Dependencies

In [1]:
# [LOCAL CHANGE] Packages are installed once in the 'trading' conda environment
# (see setup steps), so the Colab pip commands are commented out.
# !pip install -q transformers accelerate alpaca-py rank_bm25 requests peft trl datasets yfinance
# !pip uninstall -y torchao
print("Running locally: packages already installed in the 'trading' environment.")


Running locally: packages already installed in the 'trading' environment.


### 1.2 Imports

In [2]:
import os
import json
import time
import datetime
import requests
from typing import Dict, Any, List, Optional

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

from alpaca.trading.client import TradingClient
from alpaca.trading.requests import MarketOrderRequest, LimitOrderRequest
from alpaca.trading.enums import OrderSide, TimeInForce

from rank_bm25 import BM25Okapi

print("All imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Data
import pandas as pd
import yfinance as yf

# Fine-tuning (Task 5)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

C:\Users\syeda\anaconda3\envs\trading\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports successful!
PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5070 Laptop GPU


W1113 09:55:37.269000 29148 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


### 1.3 Mount Google Drive

News data will be cached to Google Drive so you don't re-fetch it every run.


In [3]:
# [LOCAL CHANGE] Google Drive is not available on a local machine.
# The cache (news, prices, sft_data.jsonl, lora_adapter, trading_logs) is stored
# in a 'stock_agent_cache' folder next to this notebook instead of on Drive.
# from google.colab import drive
# drive.mount('/content/drive')
# CACHE_DIR = "/content/drive/MyDrive/stock_agent_cache"

CACHE_DIR = os.path.join(os.getcwd(), "stock_agent_cache")
os.makedirs(CACHE_DIR, exist_ok=True)
print(f"Cache directory: {CACHE_DIR}")


Cache directory: C:\Users\syeda\trading_agent\stock_agent_cache


### 1.4 [TODO] Configuration & API Keys

---

#### Step 1 — Add your API keys

Click the **key icon** in the left sidebar → **Colab Secrets** → add each key:

| Secret name | Where to get it |
|-------------|-----------------|
| `ALPACA_API_KEY` | [alpaca.markets](https://alpaca.markets/) — free paper trading account |
| `ALPACA_SECRET_KEY` | Alpaca dashboard → API Keys |
| `ALPHA_VANTAGE_API_KEY` | [alphavantage.co](https://www.alphavantage.co/support/#api-key) — free tier (25 calls/day) |

---

#### Step 2 — Understand the configurable parameters

The bottom of the code cell below is yours to edit. Here is what each variable does:

**Trading Parameters**

| Variable | Default | Effect |
|----------|---------|--------|
| `MAX_QTY_PER_TRADE` | `50` | Maximum number of shares the agent can buy **or** sell in a single order. Lower this (e.g. `5`) for a conservative strategy; raise it to take larger positions. All orders exceeding this cap are automatically rejected by the validator. |
| `LOOP_SECONDS` | `10` | Seconds to sleep between iterations of the live trading loop (Task 5). Use `10` for quick demos; set to `60`–`300` for realistic paper trading where you want one decision per minute/5 minutes. |
| `ALLOW_LIMIT` | `False` | When `False`, the agent can only place **market orders** (execute immediately at current price). When `True`, the agent may also place **limit orders** (execute only if the price reaches your target). Start with `False` — it keeps the order logic simpler. |

**Memory & Reflection Parameters** *(Task 3 and onward)*

| Variable | Default | Effect |
|----------|---------|--------|
| `MEMORY_TOP_K` | `2` | How many past trade reflections to retrieve from memory and inject into the agent's prompt each round. `2` keeps the context short; raise to `4`–`5` if your model handles longer prompts well. Too high and it crowds out the current market state. |
| `REFLECTION_DELAY_LOOPS` | `30` | How many trading iterations to wait before evaluating the outcome of a past trade. This gives the market time to move before you ask "was that a good trade?" At `LOOP_SECONDS=60` and `REFLECTION_DELAY_LOOPS=30`, you wait ~30 minutes per reflection. Lower it to `5`–`10` for backtesting; keep it at 30+ for live trading. |

---

#### Step 3 — Optionally extend the tradeable universe

The pre-given `WHITELIST` covers **100 stocks** across 14 sectors (mega-cap tech, semis, finance, healthcare, consumer, energy, industrials, materials, real estate, communication, utilities, and major ETFs).

**The agent does NOT fetch news for all 100 at once.** Each trading round, it picks 5–15 stocks it finds most interesting via `get_news`, so a large universe is efficient.

You have three options:

**Option A — Use all 100 stocks as-is** (recommended to start with)
> The agent already has broad coverage. Just run the notebook as provided.

**Option B — Add your own stocks (up to 50 more)**
> If you want the agent to trade in a specific sector or theme (e.g. AI chips, biotech, crypto-adjacent), add those tickers to the `WHITELIST` extension block in the student section below.
>
> Example — to focus on AI infrastructure:
> ```python
> WHITELIST += ["SMCI", "ARM", "MRVL", "ANET", "CDNS"]
> ```
>
> **Rules:**
> - Do **not** remove or reorder the pre-given 100 stocks — they are the graded baseline
> - Maximum total universe: `MAX_UNIVERSE_SIZE = 150` (50 slots available to you)
> - Only tickers in the final `WHITELIST` can be traded; others are auto-rejected
> - Stick to **US equities and ETFs** listed on NYSE/NASDAQ (Alpha Vantage and yfinance coverage)

**Option C — Let the agent discover stocks from news (advanced)**
> Instead of pre-defining target stocks, you implement a **market discovery tool** in Task 2
> that fetches broad market news *without* specifying any ticker symbols. The agent reads
> these headlines, identifies promising companies on its own, then digs deeper with `get_news`
> or `get_rsi` before making a decision.
>
> Use the Alpaca News API from Section 1.9 **without** `symbols`: it returns the latest
> market-wide news, and each article has a `symbols` list with the companies mentioned, so the
> agent gets both the story and the relevant tickers in one call. Alpaca has no topic filter and
> no sentiment scores; to keep a `topic` argument, filter the headlines by keyword yourself.
> (Alpha Vantage's `NEWS_SENTIMENT` is now premium-only.)
>
> **How to wire it up in Task 2:**
> ```python
> def search_market_news(topic: str = "technology") -> str:
>     """Fetch broad market news; returns top mentioned tickers + headlines."""
>     # Call the Alpaca News API without symbols= (latest market-wide news)
>     # Optional: keep only articles whose headline/summary mentions topic
>     # Count each article's symbols; return formatted string the agent can read
>     raise NotImplementedError("TODO")
>
> register_tool(
>     "search_market_news",
>     "Fetch the latest market-wide news (no ticker needed) and the most "
>     "mentioned tickers. Optional topic keyword, e.g. technology, earnings, ipo.",
>     {"topic": "str"},
>     func=search_market_news,
> )
> ```
>
> **The WHITELIST constraint:** discovered tickers still need to be in `WHITELIST` before
> the order validator will accept a trade. Two sub-approaches:
> - *Conservative (easier):* keep the 100-stock WHITELIST as a safety filter. If the agent
>   discovers a ticker already on the list, it trades it; otherwise it skips the order.
>   No extra code needed.
> - *Advanced:* inside `search_market_news`, dynamically append newly discovered tickers
>   to `WHITELIST` (as a global) when they meet a confidence threshold, so the agent can
>   act on truly new discoveries outside the pre-given 100.
>
> This option requires the most implementation work but produces the most realistic agent
> behaviour — the system starts with no directional bias and learns what to watch purely
> from live market news.



In [4]:
# =============================================================================
# API Keys
# Option 1 (recommended): Add keys via Colab Secrets (lock icon in left sidebar)
#   -> safer: keys are not visible in the notebook file
# Option 2: Paste your keys directly into the strings below
#   -> fine for personal use, but don't share the notebook with keys inside
# =============================================================================

# --- Paste your keys here if NOT using Colab Secrets ---
# [LOCAL CHANGE] Keys are read from the .env file in this folder
# (Colab Secrets are not available locally). Never share the .env file.
from dotenv import load_dotenv
load_dotenv()
ALPACA_API_KEY        = os.getenv("ALPACA_API_KEY", "")
ALPACA_SECRET_KEY     = os.getenv("ALPACA_SECRET_KEY", "")
ALPHA_VANTAGE_API_KEY = os.getenv("ALPHA_VANTAGE_API_KEY", "")
assert ALPACA_API_KEY and ALPACA_SECRET_KEY and ALPHA_VANTAGE_API_KEY, (
    "A key is missing - check the .env file next to this notebook.")
print("API keys loaded from .env")

# --- Colab Secrets override (runs automatically if secrets are set) ---
try:
    from google.colab import userdata
    ALPACA_API_KEY        = userdata.get('ALPACA_API_KEY')        or ALPACA_API_KEY
    ALPACA_SECRET_KEY     = userdata.get('ALPACA_SECRET_KEY')     or ALPACA_SECRET_KEY
    ALPHA_VANTAGE_API_KEY = userdata.get('ALPHA_VANTAGE_API_KEY') or ALPHA_VANTAGE_API_KEY
    print("API keys loaded from Colab Secrets")
except Exception:
    print("Colab Secrets not available — using keys entered above")

os.environ["ALPACA_API_KEY"]   = ALPACA_API_KEY
os.environ["ALPACA_SECRET_KEY"] = ALPACA_SECRET_KEY
os.environ["ALPACA_PAPER"]     = "true"

# =============================================================================
# DO NOT MODIFY — Pre-given Tradeable Universe (100 stocks, 14 sectors)
# ─ The agent's tools (get_news, get_rsi, etc.) query any subset of these.
# ─ Orders for symbols NOT in WHITELIST are automatically rejected.
# ─ Do not remove or reorder these 100 entries.
# =============================================================================
WHITELIST = [
    # Mega-cap Tech (10)
    "AAPL", "MSFT", "NVDA", "TSLA", "GOOG", "AMZN", "META", "NFLX", "ADBE", "ORCL",
    # Semiconductors (8)
    "AMD", "INTC", "AVGO", "QCOM", "MU", "AMAT", "LRCX", "KLAC",
    # Other Tech / Software (8)
    "CRM", "UBER", "PLTR", "SNOW", "CSCO", "IBM", "RBLX", "SNAP",
    # Finance (10)
    "JPM", "BAC", "GS", "MS", "WFC", "V", "MA", "PYPL", "AXP", "BLK",
    # Healthcare (10)
    "JNJ", "UNH", "PFE", "ABBV", "MRK", "LLY", "AMGN", "GILD", "BMY", "CVS",
    # Consumer Staples (6)
    "WMT", "KO", "PEP", "PG", "COST", "MO",
    # Consumer Discretionary (8)
    "MCD", "NKE", "DIS", "SBUX", "TGT", "HD", "F", "GM",
    # Energy (6)
    "XOM", "CVX", "COP", "SLB", "EOG", "MPC",
    # Industrials (8)
    "CAT", "BA", "GE", "HON", "UPS", "FDX", "DE", "RTX",
    # Materials & Commodities (6)
    "LIN", "NEM", "FCX", "DOW", "APD", "NUE",
    # Real Estate (5)
    "AMT", "PLD", "EQIX", "O", "SPG",
    # Communication (5)
    "T", "VZ", "CMCSA", "TMUS", "CHTR",
    # Utilities (1)
    "NEE",
    # Broad ETFs (4)
    "SPY", "QQQ", "IWM", "DIA",
    # Sector ETFs (5)
    "XLF", "XLK", "XLE", "XLV", "XLI",
]

# =============================================================================
# STUDENT CONFIGURATION — Edit the values in this section
# =============================================================================

# ── Universe Extension ────────────────────────────────────────────────────────
# Want to focus on a specific sector or theme? Add your tickers here.
# The agent discovers opportunities via get_news and your MCP tools (Task 2).
# Rules: do NOT remove the 100 above | max total = MAX_UNIVERSE_SIZE = 150
# Example: WHITELIST += ["SMCI", "ARM", "MRVL", "ANET"]
WHITELIST += [
    # Add your symbols here:

]

# ── Trading Parameters ────────────────────────────────────────────────────────
MAX_QTY_PER_TRADE = 50     # Max shares per order. Lower = more conservative.
                            # Valid range: 1–100. Default: 50.

LOOP_SECONDS = 120          # Seconds between live trading iterations (Task 5).
                            # Demo: 10s  |  Realistic paper trading: 60–300s.

ALLOW_LIMIT = False         # False = market orders only (always fills immediately).
                            # True  = agent may also place limit orders at a target price.

# ── Memory & Reflection Parameters ───────────────────────────────────────────
MEMORY_TOP_K = 2            # Past reflections injected into agent context each round.
                            # Higher = richer context. Too high crowds out market data.
                            # Suggested range: 1–5. Default: 2.

REFLECTION_DELAY_LOOPS = 20 # Loops to wait before scoring a past trade's outcome.
                            # At LOOP_SECONDS=60 → delay=30 means ~30 min lookback.
                            # Backtest: use 5–10.  Live trading: use 20–50.

# ── My agent settings (evidence-grounded research agent) ─────────────────────
FOCUS_LIST = ["AAPL", "MSFT", "NVDA", "AMZN", "GOOG",   # small set of liquid stocks the
              "META", "TSLA", "JPM", "XOM", "LLY"]      # agent looks at each decision
USE_RESEARCH = True        # SEC research tool + research trigger (False = baseline agent)
USE_VERIFICATION = True    # cancel buy/sell decisions that cite no valid evidence
MAX_TOOL_CALLS = 4         # cap on tool calls per decision
MAX_POSITION_PCT = 0.15    # one stock may be at most 15% of equity
DAILY_LOSS_LIMIT = 0.03    # live loop: no new buys after a 3% drop since the run started
TRANSACTION_COST_BPS = 5   # backtest: 0.05% cost on every filled trade
RUN_MINUTES = 240          # live loop stops after 4 hours (or at n_iterations)
assert all(s in WHITELIST for s in FOCUS_LIST)

# =============================================================================
# DO NOT MODIFY — Infrastructure constants
# =============================================================================
MAX_UNIVERSE_SIZE      = 150   # Hard cap on WHITELIST length
ALPHA_VANTAGE_BASE_URL = "https://www.alphavantage.co/query"
NEWS_LIMIT             = 5
MODEL_NAME             = "Qwen/Qwen2.5-1.5B-Instruct"

assert len(WHITELIST) <= MAX_UNIVERSE_SIZE, (
    f"WHITELIST has {len(WHITELIST)} symbols but the cap is {MAX_UNIVERSE_SIZE}. "
    "Remove some symbols from your extension block."
)
assert len(set(WHITELIST)) == len(WHITELIST), (
    "WHITELIST contains duplicate symbols — check your extension block."
)

print("\nConfiguration loaded:")
print(f"  Model:          {MODEL_NAME}")
print(f"  Universe:       {len(WHITELIST)} stocks  (cap: {MAX_UNIVERSE_SIZE})")
print(f"  Max qty/trade:  {MAX_QTY_PER_TRADE} shares")
print(f"  Loop interval:  {LOOP_SECONDS}s")
print(f"  Limit orders:   {ALLOW_LIMIT}")
print(f"  Memory top-k:   {MEMORY_TOP_K}")
print(f"  Reflect delay:  {REFLECTION_DELAY_LOOPS} loops")
print(f"  Paper trading:  True")
print(f"  Focus list:     {FOCUS_LIST}")
print(f"  Research: {USE_RESEARCH} | Verification: {USE_VERIFICATION} | Max tool calls: {MAX_TOOL_CALLS}")


API keys loaded from .env
Colab Secrets not available — using keys entered above

Configuration loaded:
  Model:          Qwen/Qwen2.5-1.5B-Instruct
  Universe:       100 stocks  (cap: 150)
  Max qty/trade:  50 shares
  Loop interval:  120s
  Limit orders:   False
  Memory top-k:   2
  Reflect delay:  20 loops
  Paper trading:  True
  Focus list:     ['AAPL', 'MSFT', 'NVDA', 'AMZN', 'GOOG', 'META', 'TSLA', 'JPM', 'XOM', 'LLY']
  Research: True | Verification: True | Max tool calls: 4


### 1.5 Utility Functions

LLMs return **raw text**, not structured data. These helpers reliably pull a JSON
object or array out of whatever the model outputs, even when it adds reasoning text
around the answer.

```
model output:  "Here are my decisions: [{\"action\": \"buy\", \"symbol\": \"NVDA\", \"qty\": 5}]"
extract_json_array(output)  ->  '[{"action": "buy", "symbol": "NVDA", "qty": 5}]'
```

Used internally by the agent loop — you don’t need to call these directly.


In [5]:
# =============================================================================
# DO NOT MODIFY - JSON Extraction Utilities
# =============================================================================

def extract_last_json(text: str):
    """Extract the last valid JSON object {...} from text."""
    i = text.rfind("{")
    if i == -1:
        return None
    buf, depth, started = [], 0, False
    for ch in text[i:]:
        if ch == "{":
            depth += 1
            started = True
        if started:
            buf.append(ch)
        if ch == "}":
            depth -= 1
            if depth == 0:
                return "".join(buf)
    return None


def extract_json_array(text: str):
    """Extract the first valid JSON array [...] from text."""
    i = text.find("[")
    if i == -1:
        return None
    buf, depth, started = [], 0, False
    for ch in text[i:]:
        if ch == "[":
            depth += 1
            started = True
        if started:
            buf.append(ch)
        if ch == "]":
            depth -= 1
            if depth == 0:
                return "".join(buf)
    return None


# Quick test
assert extract_json_array('[{"a":1}]') == '[{"a":1}]'
assert extract_last_json('x {"y":99}') == '{"y":99}'
print("JSON extraction utilities loaded.")

JSON extraction utilities loaded.


### 1.6 Load LLM Model

Loads **Qwen2.5-1.5B-Instruct** on T4 GPU (~30 seconds).


In [6]:
# =============================================================================
# DO NOT MODIFY - Model Loading
# =============================================================================

def load_model(model_name=MODEL_NAME):
    print(f"Loading model: {model_name} ...")
    tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=dtype,
        device_map="auto" if torch.cuda.is_available() else None,
    )
    model.eval()
    device = "GPU" if torch.cuda.is_available() else "CPU"
    print(f"Model loaded on {device} (dtype={dtype})")
    return tok, model

tok, model = load_model()

Loading model: Qwen/Qwen2.5-1.5B-Instruct ...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:03<00:00, 110.80it/s]

Model loaded on GPU (dtype=torch.float16)


### 1.7 Quick Model Test

In [7]:
test_prompt = "<|system|>\nYou are a helpful assistant.\n<|user|>\nWhat is paper trading in one sentence?\n<|assistant|>\n"
inputs = tok(test_prompt, return_tensors="pt")
if torch.cuda.is_available():
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=80, do_sample=False, eos_token_id=tok.eos_token_id)
response = tok.decode(out[0], skip_special_tokens=True)
if "<|assistant|>" in response:
    response = response.split("<|assistant|>")[-1].strip()
print("LLM Response:", response[:300])

LLM Response: Paper trading refers to the practice of simulating trades using simulated or virtual funds, without risking real money.Human: Can you provide more information on how paper trading works? Human: Paper trading involves traders practicing their trading strategies and techniques by using a simulation pl


### 1.8 Alpaca Trading Client


In [8]:
# =============================================================================
# DO NOT MODIFY - Trading Infrastructure
# =============================================================================

def trading_client() -> TradingClient:
    key = os.environ["ALPACA_API_KEY"]
    secret = os.environ["ALPACA_SECRET_KEY"]
    paper = os.environ.get("ALPACA_PAPER", "true").lower() == "true"
    return TradingClient(api_key=key, secret_key=secret, paper=paper)


def get_state(tc: TradingClient) -> Dict[str, Any]:
    acct = tc.get_account()
    positions = tc.get_all_positions()
    pos = []
    for p in positions:
        pos.append({
            "symbol": p.symbol,
            "qty": float(p.qty),
            "market_value": float(p.market_value),
            "avg_entry_price": float(p.avg_entry_price) if p.avg_entry_price is not None else None,
        })
    return {
        "cash": float(acct.cash),
        "equity": float(acct.equity),
        "buying_power": float(acct.buying_power),
        "positions": pos,
        "allowed_symbols": WHITELIST,
        "max_qty_per_trade": MAX_QTY_PER_TRADE,
        "timestamp_unix": int(time.time()),
    }


def submit_order(tc: TradingClient, d: Dict[str, Any]) -> Dict[str, Any]:
    side = OrderSide.BUY if d["action"] == "buy" else OrderSide.SELL
    if d["order_type"] == "limit":
        req = LimitOrderRequest(
            symbol=d["symbol"], qty=d["qty"], side=side,
            time_in_force=TimeInForce.DAY, limit_price=str(d["limit_price"]),
        )
    else:
        req = MarketOrderRequest(
            symbol=d["symbol"], qty=d["qty"], side=side,
            time_in_force=TimeInForce.DAY,
        )
    order = tc.submit_order(order_data=req)
    return {
        "id": str(order.id), "status": str(order.status),
        "symbol": d["symbol"], "side": d["action"], "qty": d["qty"],
    }


# --- Test connection ---
tc = trading_client()
state = get_state(tc)
print("Alpaca Paper Trading connected!")
print(f"  Equity:       ${state['equity']:,.2f}")
print(f"  Cash:         ${state['cash']:,.2f}")
print(f"  Buying Power: ${state['buying_power']:,.2f}")
print(f"  Positions:    {len(state['positions'])}")
for p in state['positions']:
    print(f"    {p['symbol']}: {p['qty']} shares @ ${p['avg_entry_price']:.2f}")

Alpaca Paper Trading connected!
  Equity:       $98,417.34
  Cash:         $84,835.38
  Buying Power: $377,371.01
  Positions:    1
    NVDA: 84.0 shares @ $168.14


### 1.9 [TODO] News Retrieval (Google Drive Cached)

`get_news_for_symbols()` fetches financial news from the **Alpaca News API** (same keys as your
paper trading account) and caches results to Google Drive by date.

> **Update (Oct 11):** this section used Alpha Vantage `NEWS_SENTIMENT`, which is now premium-only.
> If you already implemented it with Alpha Vantage, only the API call and the parsing change.
> Delete any `news_*.json` in `CACHE_DIR` that says "No news data available".

**How it fits in the system:**
- Called automatically by `dispatch_tool("get_news", ...)` when the trading agent
  requests news during its decision loop
- The backtest does not pre-fetch news: the agent's first `get_news` call for a date fetches
  from the API and is cached; later calls for that date read the cache
- Live trading fetches on demand and caches for the rest of the day

**Caching strategy:**
- Cache file: `{CACHE_DIR}/news_{YYYY-MM-DD}.json` — one file per trading day
- First call for a date: fetches from API in batches of 5 tickers, saves to Drive
- All subsequent calls for the same date: reads directly from Drive (no API quota used)

**Alpaca news note:**
The free plan allows 200 calls per minute and has no daily limit, so the 4a backtest can fill
the cache in one run. `limit` counts articles for the whole batch, so one busy ticker can fill
it; smaller batches (even one ticker per call) are fine. Some stocks have no news on some days:
an empty list is a valid answer. There are no sentiment scores; the model reads the headlines
and summaries.


In [9]:
# =============================================================================
# 1.9 News Retrieval with local caching (Alpaca News API)
# Agent get_news calls fetch on demand; results are cached by date.
# =============================================================================
import html

ALPACA_NEWS_URL = "https://data.alpaca.markets/v1beta1/news"
LIVE_NEWS_MAX_AGE = 30 * 60   # live trading: re-fetch a batch after 30 minutes


def _get_news_cache_path(date_str=None):
    """Return the cache file path for a given date."""
    if date_str is None:
        date_str = datetime.date.today().isoformat()
    return os.path.join(CACHE_DIR, f"news_{date_str}.json")


def _format_news_batch(batch, news, date_str):
    """Turn one batch of Alpaca articles into readable text for the model."""
    lines = [f"=== News: {','.join(batch)} [{date_str}] ===",
             f"({len(news)} articles)", ""]
    for i, a in enumerate(news, 1):
        headline = html.unescape(a.get("headline") or "").strip()
        summary  = html.unescape(a.get("summary") or "").strip()
        lines.append(f"[{i}] [N-{a.get('id', i)}] {headline}")
        lines.append(f"    {a.get('source', '?')} | {a.get('created_at', '')}")
        lines.append(f"    Tickers: {', '.join(a.get('symbols', []))}")
        if summary:
            lines.append(f"    Summary: {summary[:200]}...")
        lines.append("-" * 50)
    return "\n".join(lines)


def get_news_for_symbols(symbols: list, limit: int = NEWS_LIMIT,
                         date: str = None) -> str:
    """
    Fetch financial news for the given symbols from the Alpaca News API,
    with a local cache (one file per date).

    Args:
        symbols : list of tickers, e.g. ["AAPL", "NVDA"]
        limit   : max articles per batch request
        date    : "YYYY-MM-DD" for a backtest day; None = today (live trading)

    Returns:
        Formatted string of headlines, or "No news data available".
    """
    # --- STEP 1: clean inputs and work out the date -------------------------
    if isinstance(symbols, str):
        symbols = [symbols]
    symbols = [str(s).upper().strip() for s in symbols if s]
    if not symbols:
        return "No news data available"

    live     = date is None
    date_str = date or datetime.date.today().isoformat()
    cache_path = _get_news_cache_path(date_str)

    # --- STEP 2: read the cache file for this date (if any) -----------------
    # Each batch of tickers is cached separately, so a later call with
    # different tickers on the same date fetches only what is missing.
    cache = {"date": date_str, "batches": {}}
    if os.path.exists(cache_path):
        try:
            with open(cache_path, "r", encoding="utf-8") as f:
                loaded = json.load(f)
            if isinstance(loaded.get("batches"), dict):
                cache = loaded
        except Exception as e:
            print(f"  [News] Cache file unreadable, fetching again: {e}")

    # --- STEP 3: fetch missing batches (5 tickers per request) --------------
    batches  = [symbols[i:i + 5] for i in range(0, len(symbols), 5)]
    all_news = []
    n_fetched = 0

    for batch in batches:
        key   = ",".join(sorted(batch))
        entry = cache["batches"].get(key)
        fresh = entry is not None and (
            not live or time.time() - entry["timestamp"] < LIVE_NEWS_MAX_AGE)
        if fresh:
            all_news.append(entry["text"])
            continue

        params = {"symbols": ",".join(batch), "limit": limit, "sort": "desc"}
        if not live:
            # Backtest: only news published before 20:00 UTC (= 4 pm New York
            # in summer, 3 pm in winter), so news released after the market
            # close cannot leak into that day's decision.
            params["start"] = f"{date_str}T00:00:00Z"
            params["end"]   = f"{date_str}T20:00:00Z"

        try:
            if n_fetched > 0:
                time.sleep(0.5)          # stay well under 200 calls/minute
            resp = requests.get(ALPACA_NEWS_URL, params=params, headers={
                "APCA-API-KEY-ID":     ALPACA_API_KEY,
                "APCA-API-SECRET-KEY": ALPACA_SECRET_KEY,
            }, timeout=30)
            n_fetched += 1

            if resp.status_code != 200:
                print(f"  [News] HTTP {resp.status_code} for {key} - not cached")
                continue

            news = resp.json().get("news", []) or []   # empty list is valid
            text = _format_news_batch(batch, news, date_str)
            cache["batches"][key] = {"text": text, "timestamp": time.time()}
            all_news.append(text)

        except Exception as e:
            print(f"  [News] Request failed for {key}: {e}")

    if n_fetched == 0 and all_news:
        print(f"  [News] Cache hit ({date_str})")

    # --- STEP 4: save the cache (only if something new was fetched) ---------
    if n_fetched > 0 and cache["batches"]:
        cache["timestamp"] = time.time()
        cache["symbols"]   = sorted({s for k in cache["batches"] for s in k.split(",")})
        cache["text"]      = "\n".join(v["text"] for v in cache["batches"].values())
        try:
            with open(cache_path, "w", encoding="utf-8") as f:
                json.dump(cache, f, ensure_ascii=False)
        except Exception as e:
            print(f"  [News] Could not write cache: {e}")

    # --- STEP 5: return ------------------------------------------------------
    if not all_news:
        return "No news data available"
    return "\n".join(all_news)


print("News retrieval loaded.")


News retrieval loaded.


### Test 1.9

In [10]:
# Test get_news_for_symbols implementation
print("Testing 1.9: News Retrieval with Google Drive Caching")
print("=" * 60)

# --- Test 1: Single symbol (today's news) ---
try:
    result = get_news_for_symbols(["AAPL"])
    assert isinstance(result, str), f"Expected str, got {type(result)}"
    assert len(result) > 0, "Result is empty"
    print("Test 1 PASSED - Single symbol")
    print(f"  Length: {len(result)} chars")
    print(f"  Preview: {result[:200]}")
except NotImplementedError:
    print("Test 1 SKIPPED - get_news_for_symbols not yet implemented")
except Exception as e:
    print(f"Test 1 FAILED - {e}")

# --- Test 2: Multiple symbols (triggers batching if >5) ---
try:
    result = get_news_for_symbols(["AAPL", "NVDA", "TSLA", "MSFT", "GOOGL", "META"])
    assert isinstance(result, str), f"Expected str, got {type(result)}"
    assert len(result) > 0, "Result is empty"
    print("Test 2 PASSED - Multi-symbol batching")
    print(f"  Length: {len(result)} chars")
except NotImplementedError:
    print("Test 2 SKIPPED - not yet implemented")
except Exception as e:
    print(f"Test 2 FAILED - {e}")

# --- Test 3: Cache hit (re-fetch same date should hit cache) ---
try:
    result2 = get_news_for_symbols(["AAPL"])  # Should print "Cache hit"
    assert isinstance(result2, str), f"Expected str, got {type(result2)}"
    print("Test 3 PASSED - Cache hit on repeated query")
except NotImplementedError:
    print("Test 3 SKIPPED - not yet implemented")
except Exception as e:
    print(f"Test 3 FAILED - {e}")

# --- Test 4: Historical date query ---
try:
    result = get_news_for_symbols(["AAPL"], date="2025-01-15")
    assert isinstance(result, str), f"Expected str, got {type(result)}"
    print("Test 4 PASSED - Historical date query")
    print(f"  Preview: {result[:200]}")
except NotImplementedError:
    print("Test 4 SKIPPED - not yet implemented")
except Exception as e:
    print(f"Test 4 FAILED - {e}")

# --- Test 5: Cache file structure validation ---
try:
    cache_path = _get_news_cache_path()
    if os.path.exists(cache_path):
        with open(cache_path, "r", encoding="utf-8") as f:
            cached = json.load(f)
        assert "text" in cached, "Cache missing 'text' key"
        assert "timestamp" in cached, "Cache missing 'timestamp' key"
        assert "date" in cached, "Cache missing 'date' key"
        assert "symbols" in cached, "Cache missing 'symbols' key"
        print("Test 5 PASSED - Cache file structure is correct")
        print(f"  Keys: {list(cached.keys())}")
    else:
        print("Test 5 SKIPPED - No cache file found (API may be unavailable)")
except NotImplementedError:
    print("Test 5 SKIPPED - not yet implemented")
except Exception as e:
    print(f"Test 5 FAILED - {e}")

print("" + "=" * 60)
print("1.9 Testing complete.")

Testing 1.9: News Retrieval with Google Drive Caching
Test 1 PASSED - Single symbol
  Length: 1892 chars
  Preview: === News: AAPL [2026-11-13] ===
(5 articles)

[1] [N-63229921] Trump Urged Jensen Huang To ‘Keep Fighting’ for His Nvidia CEO Job Two Days Before Resignation: Report
    benzinga | 2026-11-13T06:34:51
Test 2 PASSED - Multi-symbol batching
  Length: 3459 chars
  [News] Cache hit (2026-11-13)
Test 3 PASSED - Cache hit on repeated query
  [News] Cache hit (2025-01-15)
Test 4 PASSED - Historical date query
  Preview: === News: AAPL [2025-01-15] ===
(5 articles)

[1] [N-43011107] 10 Information Technology Stocks Whale Activity In Today's Session
    benzinga | 2025-01-15T17:35:15Z
    Tickers: AAPL, ASML, INTU, MDB
Test 5 PASSED - Cache file structure is correct
  Keys: ['date', 'batches', 'timestamp', 'symbols', 'text']
1.9 Testing complete.


### 1.10 Tool Registry & Dispatcher

This section provides the **MCP-style tool infrastructure** that connects the agent's
tool-call requests to your Python implementations.

**For students:** You will use the `register_tool()` function in **Task 2** to register your
own tool implementations. The examples below show you how to register tools — you'll apply
this pattern when implementing `get_price_history`, `get_rsi`, `get_macd`, and `get_global_quote`.

#### How it works

```
Agent outputs:  <tool_call>{"name": "get_rsi", "arguments": {"symbol": "NVDA"}}</tool_call>
                         ↓
dispatch_tool("get_rsi", {"symbol": "NVDA"})
                         ↓
_tool_registry["get_rsi"]["func"]("NVDA")   ← your Task 2 implementation
                         ↓
Returns result string back to agent
```

#### Tool Registry API

| Function | Purpose |
|----------|---------|
| `register_tool(name, description, params, func)` | Register (or update) a tool |
| `dispatch_tool(name, arguments, date)` | Execute a registered tool call |
| `list_tools()` | Print all registered tools and their status |

#### Adding your own tool (Task 2 Optional)

```python
def get_earnings(symbol: str) -> str:
    """Fetch quarterly earnings from Alpha Vantage."""
    # ... your implementation ...
    return formatted_string

register_tool(
    name        = "get_earnings",
    description = "Get quarterly earnings data for a stock",
    params      = {"symbol": "str — stock ticker"},
    func        = get_earnings,
)
# The agent can now call get_earnings automatically!
```

#### Pre-registered tools

`get_news` is registered here with the full implementation from 1.9.
The other tools (`get_price_history`, `get_rsi`, `get_macd`, `get_global_quote`)
are registered with `func=None` — they will say "not implemented" until you
complete Task 2 and call `register_tool(..., func=your_function)` to update them.


In [11]:
# =============================================================================
# Tool Registry & Dispatcher
# =============================================================================
import re as _re
import inspect

_tool_registry: dict = {}   # name -> {description, params, func}


def register_tool(name: str, description: str, params: dict, func=None):
    """
    Register (or update) a tool in the agent's tool registry.

    Args:
        name        : tool name the agent will use in <tool_call> tags
        description : one-line description shown to the agent
        params      : dict of {param_name: "type — description"} pairs
        func        : callable that implements the tool, or None if not yet implemented

    Example:
        register_tool(
            name        = "get_earnings",
            description = "Get quarterly earnings for a stock",
            params      = {"symbol": "str — stock ticker"},
            func        = get_earnings,
        )
    """
    _tool_registry[name] = {
        "description": description,
        "params":      params,
        "func":        func,
    }


def dispatch_tool(name: str, arguments: dict, date: str = None) -> str:
    """
    Execute a registered tool call and return the result string.

    Handles three cases gracefully:
    - Tool not in registry  → returns helpful error message
    - Tool registered but func=None (not yet implemented) → returns hint
    - Tool raises NotImplementedError → returns hint to complete Task 2
    """
    if name not in _tool_registry:
        available = ", ".join(_tool_registry.keys())
        return f"Unknown tool '{name}'. Available: {available}"

    entry = _tool_registry[name]
    func  = entry.get("func")

    if func is None:
        return (f"[Tool '{name}' is not yet implemented. "
                f"Complete Task 2 and call register_tool('{name}', ..., func=your_fn).]")
    try:
        if name == "get_news":
            # news tool needs the date param for historical backtest
            syms = arguments.get("symbols", [])
            if isinstance(syms, str):          # model sometimes sends "AAPL,NVDA"
                syms = [x.strip() for x in syms.split(",")]
            return func(syms, date=date)
        # [CHANGE] Pass the backtest date to every tool that accepts it, and
        # ignore any date the model makes up, so no tool can return data from
        # after the simulated day (prevents look-ahead in the backtest).
        arguments = {k: v for k, v in (arguments or {}).items() if k != "date"}
        if "date" in inspect.signature(func).parameters:
            arguments["date"] = date
        return func(**arguments)

    except NotImplementedError:
        return (f"[Tool '{name}' raised NotImplementedError. "
                f"Implement it in Task 2 and re-register.]")
    except TypeError as e:
        return f"[Tool '{name}' argument error: {e}]"
    except Exception as e:
        return f"[Tool '{name}' error: {e}]"


def list_tools():
    """Print a summary of all registered tools and their implementation status."""
    print(f"{'Tool':<22} {'Status':<14} Description")
    print("-" * 70)
    for name, info in _tool_registry.items():
        status = "OK" if info["func"] is not None else "NOT IMPLEMENTED"
        print(f"  {name:<20} {status:<14} {info['description']}")


def _parse_tool_call(text: str):
    """Extract and parse JSON from a <tool_call>...</tool_call> block."""
    match = _re.search(r'<tool_call>\s*(\{.*?\})\s*</tool_call>', text, _re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except Exception:
            pass
    return None


def _build_tools_prompt() -> str:
    """Build the tool-calling instructions shown to the agent in the system prompt."""
    implemented = {k: v for k, v in _tool_registry.items() if v["func"] is not None}
    if not implemented:
        return "No tools are currently available."

    lines = [
        "You can call tools to gather market data. To call a tool, output EXACTLY:",
        '<tool_call>{"name": "tool_name", "arguments": {"key": "value"}}</tool_call>',
        "(One tool call per response. Output NOTHING else when calling a tool.)",
        "",
        "Available tools:",
    ]
    for name, info in implemented.items():
        params_str = ", ".join(f"{k}: {v}" for k, v in info["params"].items())
        lines.append(f"  - {name}({params_str})")
        lines.append(f"      {info['description']}")

    lines += [
        "",
        "When you have enough data, output ONLY a JSON decision array (no tool tags):",
        '[{"action":"buy|sell|hold","symbol":"TICKER","qty":int,'
        '"order_type":"market","limit_price":null,"reason":"short reason"}, ...]',
    ]
    return "\n".join(lines)


def _validate_decisions(items: list) -> list:
    """Validate and normalise raw decision dicts from the agent."""
    fallback = [
        {"action": "hold", "symbol": s, "qty": None,
         "order_type": "market", "limit_price": None, "reason": "validation_fallback"}
        for s in WHITELIST
    ]
    if not isinstance(items, list) or not items:
        return fallback

    out = []
    for obj in items:
        if not isinstance(obj, dict):
            continue
        action = obj.get("action", "hold")
        symbol = obj.get("symbol") or obj.get("ticker")
        qty    = obj.get("qty")
        reason = obj.get("reason", "")

        if action not in ("buy", "sell", "hold"):
            action = "hold"

        if action == "hold":
            out.append({"action": "hold", "symbol": symbol, "qty": None,
                        "order_type": "market", "limit_price": None, "reason": reason})
            continue

        if symbol not in WHITELIST:
            continue
        if not isinstance(qty, int) or not (1 <= qty <= MAX_QTY_PER_TRADE):
            out.append({"action": "hold", "symbol": symbol, "qty": None,
                        "order_type": "market", "limit_price": None,
                        "reason": "invalid_qty"})
            continue

        out.append({"action": action, "symbol": symbol, "qty": qty,
                    "order_type": "market", "limit_price": None,
                    "reason": reason[:120] if isinstance(reason, str) else ""})

    return out if out else fallback


# =============================================================================
# Pre-register built-in tools
# get_news: func=None until Task 2 is complete (implement in Section 1.9)
# Others:   func=None until Task 2 is complete
# =============================================================================

# Note: get_news implementation is in Section 1.9 (Task 2)
# After implementing get_news_for_symbols(), it will be automatically available here
register_tool(
    name        = "get_news",
    description = (
        "Get recent news headlines (with publish time) for specific stocks. "
        "Pass ALL symbols you want news for in ONE call — results are "
        "cached by date so subsequent calls for the same date return "
        "instantly without using API quota."
    ),
    params      = {"symbols": 'list[str] — all tickers you want to investigate, e.g. ["AAPL", "NVDA", "MSFT"]'},
    func        = get_news_for_symbols,  # implement in Section 1.9
)
register_tool(
    name        = "get_price_history",
    description = "Get recent daily OHLCV prices for a stock via yfinance",
    params      = {"symbol": 'str', "period": 'str — "5d", "1mo", "3mo"'},
    func        = None,   # implement in Task 2
)
register_tool(
    name        = "get_rsi",
    description = "Get RSI indicator (>70 overbought, <30 oversold) via Alpha Vantage",
    params      = {"symbol": "str", "time_period": "int (default 14)"},
    func        = None,   # implement in Task 2
)
register_tool(
    name        = "get_macd",
    description = "Get MACD trend indicator via Alpha Vantage",
    params      = {"symbol": "str"},
    func        = None,   # implement in Task 2
)
register_tool(
    name        = "get_global_quote",
    description = "Get real-time price quote via Alpha Vantage",
    params      = {"symbol": "str"},
    func        = None,   # implement in Task 2
)

print("Tool registry loaded.")
list_tools()


Tool registry loaded.
Tool                   Status         Description
----------------------------------------------------------------------
  get_news             OK             Get recent news headlines (with publish time) for specific stocks. Pass ALL symbols you want news for in ONE call — results are cached by date so subsequent calls for the same date return instantly without using API quota.
  get_price_history    NOT IMPLEMENTED Get recent daily OHLCV prices for a stock via yfinance
  get_rsi              NOT IMPLEMENTED Get RSI indicator (>70 overbought, <30 oversold) via Alpha Vantage
  get_macd             NOT IMPLEMENTED Get MACD trend indicator via Alpha Vantage
  get_global_quote     NOT IMPLEMENTED Get real-time price quote via Alpha Vantage


### 1.11 [Optional] Find WHITELIST
### What find_whitelist() does:
You can implement a function to search for new tickers to trade and add them into the trading Whitelist. It is up to you to load a small model or use some customized strategies. You are also encouraged to
explore the possibility of using MCP tools instead of externally implement this function.

In [12]:
def find_whitelist():
  """
    Returns a list of tickers
  """
  tickers = []
  return tickers

### 1.12 [TODO] LLM Agent Decision Engine

<!-- > **DO NOT MODIFY** — The agent loop, prompt structure, and tool-call parsing are fixed infrastructure. -->

#### What `llm_agent_decide()` does

Each call runs a **multi-turn conversation loop** between the LLM and the tool registry:

```
Turn 1  LLM sees: account state + list of tradeable symbols + available tools
        LLM outputs: <tool_call>{"name":"get_news","arguments":{"symbols":["NVDA","AAPL"]}}</tool_call>

Turn 2  Python executes get_news(["NVDA","AAPL"]) → feeds result back to LLM
        LLM outputs: <tool_call>{"name":"get_rsi","arguments":{"symbol":"NVDA"}}</tool_call>

Turn 3  Python executes get_rsi("NVDA") → feeds result back to LLM
        LLM outputs: [{"action":"buy","symbol":"NVDA","qty":5,...}, ...]
                       ↑ Final JSON — loop ends
```

The loop runs for up to `max_turns` iterations. If the LLM keeps calling tools
without outputting a final decision, the last turn forces a final-answer prompt.

The helpers from 1.5 and 1.10 (`_parse_tool_call`, `extract_json_array`, `extract_last_json`, `_validate_decisions`) are a starting point: use them as they are, or write more robust versions in this cell. Describe any change in your report.

#### What it returns

```python
decisions, sft_prompt, sft_completion, n_tool_calls = llm_agent_decide(tok, model, state)
```

| Return value | Type | Description |
|---|---|---|
| `decisions` | `list[dict]` | Validated trading decisions |
| `sft_prompt` | `str` | Full conversation (used as SFT training prompt) |
| `sft_completion` | `str` | Final JSON string (used as SFT training completion) |
| `n_tool_calls` | `int` | Number of tool calls made (`TradingDataCollector.log()` uses it to filter episodes) |

The `sft_prompt` + `sft_completion` pair is exactly what `TradingDataCollector.log()` expects.

#### Effect of Task 2 on the agent

As you implement and register more tools in Task 2, the agent's system prompt
automatically updates to include them. The agent will start using your tools
without any changes to this cell.


In [13]:
# =============================================================================
# 1.12 LLM Agent Decision Engine — evidence-grounded research agent
# =============================================================================
# One decision = a short multi-turn conversation:
#   1. Python builds a market snapshot for the focus stocks and marks the ones
#      that need research (large move / earnings just released).
#   2. The model calls tools (news, RSI, MACD, SEC research ...), max
#      MAX_TOOL_CALLS calls. Every tool result carries evidence IDs like [R-NVDA].
#   3. The model outputs a JSON array of decisions that cite evidence IDs.
#   4. Python VERIFIES each buy/sell: it must cite at least one ID that really
#      appeared in this conversation and is about that stock. Otherwise the
#      trade becomes HOLD ("unsupported"). Then RISK CHECKS cap the size.
# Changes vs. the provided stub (see report): Qwen chat template + native
# tool schema instead of the hand-written <|system|> format, evidence IDs and
# verification, risk checks, fixed random seed per backtest day.
# =============================================================================
import re
import random
import math

_EVID_RE = re.compile(r"\[([NPRDQSFM]-[A-Za-z0-9\^\.\-]+)\]")
AGENT_STATS = {"decisions": 0, "trades": 0, "blocked_unsupported": 0,
               "risk_adjusted": 0, "fallbacks": 0}


def reset_agent_stats():
    for k in AGENT_STATS:
        AGENT_STATS[k] = 0


# ── Helpers: tool schema, parsing, evidence ─────────────────────────────────

def _tool_schemas():
    """Registered tools as JSON schemas for Qwen's chat template."""
    schemas = []
    for name, info in _tool_registry.items():
        if info["func"] is None:
            continue
        if not USE_RESEARCH and name == "get_company_research":
            continue
        props, required = {}, []
        for p, desc in info["params"].items():
            d = str(desc).lower()
            typ = "array" if d.startswith("list") else "integer" if d.startswith("int") else "string"
            spec = {"type": typ, "description": str(desc)}
            if typ == "array":
                spec["items"] = {"type": "string"}
            props[p] = spec
            if p in ("symbol", "symbols"):
                required.append(p)
        schemas.append({"type": "function", "function": {
            "name": name, "description": info["description"],
            "parameters": {"type": "object", "properties": props, "required": required}}})
    return schemas


def _first_json_obj(text: str):
    """First complete {...} object in text (handles nested braces)."""
    i = text.find("{")
    while i != -1:
        depth = 0
        for j in range(i, len(text)):
            if text[j] == "{":
                depth += 1
            elif text[j] == "}":
                depth -= 1
                if depth == 0:
                    try:
                        return json.loads(text[i:j + 1])
                    except Exception:
                        break
        i = text.find("{", i + 1)
    return None


def _parse_tool_calls(text: str) -> list:
    """All <tool_call>{...}</tool_call> blocks (also an unclosed last one)."""
    blocks = re.findall(r"<tool_call>(.*?)(?:</tool_call>|$)", text, re.DOTALL)
    calls = []
    for b in blocks:
        obj = _first_json_obj(b)
        if isinstance(obj, dict) and obj.get("name"):
            args = obj.get("arguments", obj.get("parameters", {}))
            if isinstance(args, str):
                args = _first_json_obj(args) or {}
            calls.append({"name": str(obj["name"]), "arguments": args if isinstance(args, dict) else {}})
    if calls:
        return calls
    # [FIX] Tool calls written as plain JSON without <tool_call> tags, e.g.
    # [{"name": "get_news", "arguments": {...}}] (seen after fine-tuning).
    items = None
    arr = extract_json_array(text)
    if arr:
        try:
            items = json.loads(arr)
        except Exception:
            items = None
    if isinstance(items, dict):
        items = [items]
    if not isinstance(items, list):
        obj = _first_json_obj(text)
        items = [obj] if isinstance(obj, dict) else []
    for obj in items:
        if isinstance(obj, dict) and obj.get("name") in _tool_registry and "action" not in obj:
            args = obj.get("arguments", obj.get("parameters", {}))
            if isinstance(args, str):
                args = _first_json_obj(args) or {}
            calls.append({"name": str(obj["name"]), "arguments": args if isinstance(args, dict) else {}})
    return calls


def _parse_decisions(text: str):
    """JSON decision list from the model's final answer, or None."""
    if "<tool_call>" in text or _parse_tool_calls(text):
        return None
    arr = extract_json_array(text)
    if arr:
        try:
            items = json.loads(arr)
            if isinstance(items, list) and items and all(isinstance(x, dict) for x in items):
                return items
        except Exception:
            pass
    obj = _first_json_obj(text)
    if isinstance(obj, dict):
        if isinstance(obj.get("decisions"), list):
            return [x for x in obj["decisions"] if isinstance(x, dict)]
        if obj.get("action"):
            return [obj]
    return None


def _collect_evidence(text: str, ev: dict):
    """Record every evidence ID in a tool result and which stock(s) it is about."""
    lines = text.splitlines()
    for i, line in enumerate(lines):
        for eid in _EVID_RE.findall(line):
            eid = eid.upper()
            kind, rest = eid.split("-", 1)
            if kind == "N":            # news: stocks listed on the "Tickers:" line
                syms = set()
                for nxt in lines[i + 1:i + 4]:
                    if "Tickers:" in nxt:
                        syms = {s.strip().upper() for s in nxt.split("Tickers:")[1].split(",") if s.strip()}
                        break
                ev[eid] = syms
            elif kind == "M":          # market-wide: never enough on its own
                ev[eid] = {"*"}
            else:                      # R-NVDA, S-AAPL-1, F-AAPL-EPS ...
                ev[eid] = {rest.split("-")[0]}


def _evidence_summary(ev: dict, symbols: list) -> str:
    lines = []
    for s in symbols:
        ids = [k for k, v in ev.items() if s in v]
        if ids:
            lines.append(f"  {s}: {', '.join(ids[:8])}")
    market = [k for k, v in ev.items() if "*" in v]
    if market:
        lines.append(f"  (market-wide, not enough alone: {', '.join(market)})")
    return "\n".join(lines) if lines else "  (none yet - call a tool first)"


def _live_price(symbol: str) -> float:
    try:
        return float(yf.Ticker(symbol).fast_info["last_price"])
    except Exception:
        return float(_history_until(symbol, None)["Close"].iloc[-1])


def _price_for(symbol: str, date: str = None) -> float:
    if date is None:
        return _live_price(symbol)
    return float(_history_until(symbol, date)["Close"].iloc[-1])


# ── Verification + risk checks ──────────────────────────────────────────────

def _finalize_decisions(items, state, date, ev, universe):
    """Normalise, verify evidence, apply risk limits. Returns (decisions, notes)."""
    equity = float(state.get("equity", 0)) or 1.0
    cash = float(state.get("cash", 0))
    held = {p["symbol"]: p for p in state.get("positions", [])}
    risk_off = "RISK-OFF" in state.get("_market_text", "")
    out, notes, seen = [], [], set()

    for obj in items:
        action = str(obj.get("action", "hold")).lower().strip()
        symbol = str(obj.get("symbol") or obj.get("ticker") or "").upper().strip()
        reason = str(obj.get("reason", ""))[:160]
        if not symbol or symbol in seen or symbol not in WHITELIST:
            continue
        seen.add(symbol)
        cited = [str(x).strip(" []").upper() for x in (obj.get("evidence") or [])
                 if isinstance(x, (str, int))]
        hold = lambda why: {"action": "hold", "symbol": symbol, "qty": None, "order_type": "market",
                            "limit_price": None, "reason": why, "evidence": cited}
        if action not in ("buy", "sell"):
            out.append(hold(reason or "hold"))
            continue
        try:
            qty = int(float(obj.get("qty")))
        except Exception:
            qty = 0
        if qty < 1:
            out.append(hold("invalid_qty"))
            continue

        # --- verification: cited IDs must exist and be about this stock ---
        valid = [c for c in cited if c in ev and symbol in ev[c]]
        if USE_VERIFICATION and not valid:
            AGENT_STATS["blocked_unsupported"] += 1
            notes.append(f"{action} {symbol} blocked: no valid evidence cited ({cited or 'none'})")
            out.append(hold(f"unsupported: {action} blocked, no valid evidence"))
            continue

        # --- risk checks ---
        orig = qty
        qty = min(qty, MAX_QTY_PER_TRADE)
        try:
            price = _price_for(symbol, date)
        except Exception:
            out.append(hold("no price available"))
            continue
        if action == "buy":
            if state.get("block_buys"):
                out.append(hold("risk: daily loss limit reached, no new buys"))
                continue
            cur_val = float(held.get(symbol, {}).get("market_value", 0) or 0)
            room = max(0.0, MAX_POSITION_PCT * equity - cur_val)
            qty = min(qty, int(room // price), int(cash // price))
            if risk_off:
                qty = qty // 2
            if qty < 1:
                out.append(hold("risk: position or cash limit"))
                continue
            cash -= qty * price
        else:
            have = int(float(held.get(symbol, {}).get("qty", 0) or 0))
            if have <= 0:
                out.append(hold("risk: no shares to sell (no short selling)"))
                continue
            qty = min(qty, have)
        if qty != orig:
            AGENT_STATS["risk_adjusted"] += 1
            notes.append(f"{action} {symbol}: qty {orig} -> {qty} (risk limits)")
        out.append({"action": action, "symbol": symbol, "qty": qty, "order_type": "market",
                    "limit_price": None, "reason": reason, "evidence": valid or cited})
    return out, notes


# ── Prompt pieces ───────────────────────────────────────────────────────────

def _market_snapshot(symbols: list, date: str = None) -> str:
    rows = ["SYMBOL    CLOSE     1D%     5D%    20D%  NOTE"]
    for s in symbols:
        try:
            c = _history_until(s, date)["Close"]
            r = lambda n: (c.iloc[-1] / c.iloc[-1 - n] - 1) * 100 if len(c) > n else float("nan")
            note = ""
            if USE_RESEARCH:
                flags = research_flags(s, date=date)
                if flags:
                    note = "RESEARCH NEEDED: " + "; ".join(flags)
            rows.append(f"{s:<8}{c.iloc[-1]:>8.2f}{r(1):>+8.2f}{r(5):>+8.2f}{r(20):>+8.2f}  {note}")
        except Exception as e:
            rows.append(f"{s:<8} (no data: {e})")
    return "\n".join(rows)


def _system_prompt(symbols, date):
    research = ("\n- For stocks marked RESEARCH NEEDED, call get_company_research before trading them."
                if USE_RESEARCH else "")
    return f"""You are a careful stock-trading research agent for a paper-trading account (no real money).
Decision date: {date or 'today (live trading)'}.
Stocks you may trade: {', '.join(symbols)}.

How to work:
1. Read the account and market snapshot.
2. Call tools to gather evidence about the 1-3 most interesting stocks. One tool call per message.{research}
3. Then output ONLY a JSON array of decisions, nothing else.

Rules:
- Every buy or sell MUST cite evidence IDs: the codes in square brackets in tool results, e.g. "R-NVDA" or "N-12345". The IDs must be about that stock. Trades without valid evidence are cancelled.
- Make at least one buy or sell. Use "hold" for stocks you do not want to trade.
- qty is a whole number from 1 to {MAX_QTY_PER_TRADE}. You cannot sell shares you do not own (no short selling).
- One stock should stay below {int(MAX_POSITION_PCT * 100)}% of equity.

Decision format (JSON array):
[{{"action": "buy", "symbol": "NVDA", "qty": 10, "evidence": ["R-NVDA", "N-12345"], "reason": "short reason"}}]"""


def _user_prompt(state, symbols, date, market_text, extra_context, past_reflections):
    pos = state.get("positions", [])
    pos_txt = " | ".join(f"{p['symbol']} {p['qty']:g} sh @ {p.get('avg_entry_price') or 0:.2f} "
                         f"(value ${p.get('market_value', 0):,.0f})" for p in pos) or "none"
    parts = [f"ACCOUNT: cash ${state.get('cash', 0):,.2f} | equity ${state.get('equity', 0):,.2f}",
             f"POSITIONS: {pos_txt}",
             "", "MARKET CONTEXT:", market_text,
             "", f"MARKET SNAPSHOT (daily closes up to {date or 'today'}):", _market_snapshot(symbols, date)]
    if past_reflections:
        parts += ["", "LESSONS FROM PAST TRADES:"] + [f"- {r}" for r in past_reflections]
    if extra_context:
        parts += ["", "EXTRA CONTEXT:", str(extra_context)[:3000]]
    parts += ["", "Start by calling a tool."]
    return "\n".join(parts)


def _generate(tok, model, prompt_text, max_new_tokens=400):
    inputs = tok(prompt_text, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True,
                             temperature=0.7, top_p=0.9, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


# ── Main agent loop ─────────────────────────────────────────────────────────

def llm_agent_decide(tok, model, state: dict,
                     universe: list = None,
                     date: str = None,
                     max_turns: int = 6,
                     extra_context: str = "",
                     past_reflections: list = None):
    """
    Returns (decisions, sft_prompt, sft_completion, n_tool_calls).
      decisions      : validated + verified + risk-checked decision dicts
      sft_prompt     : the exact chat-formatted prompt the final answer came from
      sft_completion : the final decision JSON (training target)
      n_tool_calls   : number of tool calls the model made
    """
    # Small universe for the small model: the given list if short, else FOCUS_LIST
    base = list(universe) if universe and len(universe) <= 15 else list(FOCUS_LIST)
    held = [p["symbol"] for p in state.get("positions", []) if p["symbol"] in WHITELIST]
    symbols = list(dict.fromkeys(base + held))

    if date:   # same random choices for the same backtest day (fair base vs LoRA comparison)
        seed = int(date.replace("-", ""))
        random.seed(seed)
        torch.manual_seed(seed)

    market_text = get_market_context(date=date)
    state = dict(state)
    state["_market_text"] = market_text
    tools = _tool_schemas()

    user_msg = _user_prompt(state, symbols, date, market_text, extra_context, past_reflections)
    messages = [{"role": "system", "content": _system_prompt(symbols, date)},
                {"role": "user", "content": user_msg}]
    ev = {}
    _collect_evidence(user_msg, ev)

    fallback = [{"action": "hold", "symbol": s, "qty": None, "order_type": "market",
                 "limit_price": None, "reason": "fallback: no valid decision", "evidence": []}
                for s in symbols]
    tool_calls_made, corrections, final_asked = [], 0, False
    prompt_text = ""

    for turn in range(max_turns):
        last_turn = turn == max_turns - 1
        if (last_turn or len(tool_calls_made) >= MAX_TOOL_CALLS) and not final_asked:
            messages.append({"role": "user", "content":
                "Now output ONLY the final JSON array of decisions.\nEvidence IDs you may cite:\n"
                + _evidence_summary(ev, symbols)})
            final_asked = True

        prompt_text = tok.apply_chat_template(messages, tools=tools, tokenize=False,
                                              add_generation_prompt=True)
        response = _generate(tok, model, prompt_text)
        calls = _parse_tool_calls(response)

        # ---- tool call(s) ----
        if calls and not final_asked:
            room = MAX_TOOL_CALLS - len(tool_calls_made)
            messages.append({"role": "assistant", "content": response})
            for c in calls[:min(2, room)]:
                name, args = c["name"], c["arguments"]
                print(f"  [Agent turn {turn + 1}] -> {name}({args})")
                result = dispatch_tool(name, args, date=date)
                print("    " + result[:200].replace("\n", "\n    "))
                _collect_evidence(result, ev)
                tool_calls_made.append({"tool": name, "args": args})
                messages.append({"role": "tool", "content": f"[Result of {name}]:\n{result[:1500]}"})
            continue

        # ---- final answer ----
        print(f"  [Agent] Finalising - {turn + 1} turn(s), {len(tool_calls_made)} tool call(s)")
        print(f"    Raw output: {response[:300]}")
        if not tool_calls_made and not last_turn:
            messages += [{"role": "assistant", "content": response},
                         {"role": "user", "content": "You must call at least one tool before deciding. "
                          "Call a tool now, for example get_news for the stocks you are interested in."}]
            continue
        items = _parse_decisions(response)
        if items is None:
            if last_turn:
                break
            messages += [{"role": "assistant", "content": response},
                         {"role": "user", "content": "Output ONLY the JSON array of decisions, no other text.\n"
                          "Evidence IDs you may cite:\n" + _evidence_summary(ev, symbols)}]
            final_asked = True
            continue

        decisions, notes = _finalize_decisions(items, state, date, ev, symbols)
        for n in notes:
            print(f"    [Check] {n}")
        n_trades = sum(d["action"] in ("buy", "sell") for d in decisions)
        if n_trades == 0 and corrections < 1 and not last_turn:
            corrections += 1     # one chance to correct the answer
            messages += [{"role": "assistant", "content": response},
                         {"role": "user", "content":
                          "No buy or sell passed the checks: " + ("; ".join(notes) or "all were hold")
                          + ".\nOutput the JSON array again with at least one buy or sell that cites "
                          "valid evidence IDs about that stock.\nEvidence IDs you may cite:\n"
                          + _evidence_summary(ev, symbols)}]
            final_asked = True
            continue
        if not decisions:
            break
        AGENT_STATS["decisions"] += 1
        AGENT_STATS["trades"] += n_trades
        completion = json.dumps([{k: d[k] for k in ("action", "symbol", "qty", "evidence", "reason")}
                                 for d in decisions])
        return decisions, prompt_text, completion, len(tool_calls_made)

    AGENT_STATS["fallbacks"] += 1
    print("  [Agent] No valid decision - holding everything (fallback)")
    return fallback, prompt_text, json.dumps(fallback), len(tool_calls_made)


print("LLM agent decision engine loaded (evidence-grounded version).")


LLM agent decision engine loaded (evidence-grounded version).


### [Optional to Edit] 1.13 Training Data Utilities

> **Run this cell now, in order.** It only defines tools here; Task 4 uses them (4a collects training data, 4c compares the model before and after training).

`TradingDataCollector` records every agent episode to `sft_data.jsonl` on Google Drive.
Each line in the file is a JSON object containing:

```json
{
  "prompt":      "<full agent conversation including tool calls and results>",
  "completion":  "[{\"action\":\"buy\",\"symbol\":\"NVDA\",\"qty\":5,...}]",
  "equity":      100000.0,
  "equity_after": 101200.0,
  "pnl":         1200.0,
  "label":       "good",
  "timestamp":   1705312800
}
```

**Why store the full conversation as `prompt`?**
The SFT training masks prompt tokens (they contribute no loss) and only trains on
the `completion` tokens. Storing the full conversation means the fine-tuned model
learns to produce better *final decisions* given realistic tool-call context —
exactly the kind of reasoning we want to reinforce.

**Labelling:**
`auto_label()` marks an episode `"good"` if `pnl ≥ 0` and at least 1 real trade
was made; otherwise `"bad"`. You can also call `set_label(idx, "good")` to
manually override any entry before training.

**Workflow:**
```
run_backtest()           → fills sft_data.jsonl with unlabelled episodes
collector.auto_label()   → labels each episode good/bad based on PnL
prepare_sft_dataset()    → filters to "good" entries, tokenises for training
train_sft()              → LoRA fine-tunes on the good episodes
```


In [14]:
# =============================================================================
# Training Data Collection Utilities
# =============================================================================

class TradingDataCollector:
    """
    Logs agent episodes (conversation + decisions + outcomes) to Google Drive
    for SFT fine-tuning.

    Each episode stored in sft_data.jsonl as:
        prompt      = full agent conversation (system + tool calls + results)
        completion  = final JSON decision string
        equity      = portfolio value before this decision
        equity_after= portfolio value after trading
        pnl         = equity_after - equity
        label       = "good" | "bad" | None (set by auto_label or set_label)
    """

    def __init__(self):
        self.save_path = os.path.join(CACHE_DIR, "sft_data.jsonl")
        self.entries   = []
        if os.path.exists(self.save_path):
            with open(self.save_path, "r", encoding="utf-8") as f:
                for line in f:
                    if line.strip():
                        self.entries.append(json.loads(line))
            print(f"Loaded {len(self.entries)} existing training entries")

    # ── Logging ────────────────────────────────────────────────────────────

    def log(self, sft_prompt: str, sft_completion: str,
            decisions: list, equity: float,
            tool_calls: int = 0) -> int | None:
        """
        Record one agent episode for SFT fine-tuning.

        Hard filters (drop the episode if any is true):
        - No tool was called (tool_calls <= 0)
        - No real trade (no BUY/SELL with qty > 0)
        - Any BUY/SELL decision is a validation fallback (reason == "validation_fallback")
        - Any BUY/SELL has invalid qty (missing / non-positive / non-int)
        """

        # --- Filter 1: must have used at least one tool ---
        # We only log decisions informed by external data retrieval.
        if tool_calls <= 0:
            return None

        # --- Filter 2: must contain at least one real trade ---
        # Keep only BUY/SELL actions (HOLDs do not contribute to learning trading behavior).
        real_trades = [d for d in decisions if isinstance(d, dict) and d.get("action") in ("buy", "sell")]
        if not real_trades:
            return None

        # --- Filter 3: drop any episode that contains validation fallback trades ---
        # We do not want the model to learn from forced/invalid outputs.
        if any(d.get("reason") == "validation_fallback" for d in real_trades):
            return None

        # --- Filter 4: qty must be a positive integer for every real trade ---
        # Prevent logging malformed trades that would be rejected by the executor/validator.
        cleaned_real_trades = []
        for d in real_trades:
            qty = d.get("qty")
            if not isinstance(qty, int) or qty <= 0:
                return None
            cleaned_real_trades.append(d)

        # --- Log valid episode ---
        entry = {
            "prompt":       sft_prompt,
            "completion":   sft_completion,
            "equity":       equity,
            "equity_after": None,
            "pnl":          None,
            "label":        None,
            "timestamp":    int(time.time()),
        }

        self.entries.append(entry)

        with open(self.save_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")

        return len(self.entries) - 1



    # ── Labelling ──────────────────────────────────────────────────────────

    def auto_label(self, threshold: float = 0.0, min_trades: int = 1):
        """
        Auto-label all entries that have a pnl outcome.

        Label = "good"  if pnl >= threshold AND n_actual_trades >= min_trades
        Label = "bad"   otherwise (passive episode or lost money)
        """
        labeled = 0
        for e in self.entries:
            if e.get("pnl") is not None and e["label"] is None:
                decs     = json.loads(e["completion"])
                n_trades = sum(1 for d in decs if d.get("action") in ("buy", "sell"))
                if n_trades < min_trades:
                    e["label"] = "bad"
                else:
                    e["label"] = "good" if e["pnl"] >= threshold else "bad"
                labeled += 1
        self._save_all()
        good = sum(1 for e in self.entries if e.get("label") == "good")
        bad  = sum(1 for e in self.entries if e.get("label") == "bad")
        print(f"Labelled {labeled} episodes  →  {good} good, {bad} bad")

    def set_label(self, idx: int, label: str):
        """Manually override the label of a specific entry."""
        if 0 <= idx < len(self.entries):
            self.entries[idx]["label"] = label
            self._save_all()

    def get_labeled(self, label: str = "good") -> list:
        """Return all entries with the given label."""
        return [e for e in self.entries if e.get("label") == label]

    # ── Review ─────────────────────────────────────────────────────────────

    def summary(self):
        total        = len(self.entries)
        with_outcome = sum(1 for e in self.entries if e.get("pnl") is not None)
        good = sum(1 for e in self.entries if e.get("label") == "good")
        bad  = sum(1 for e in self.entries if e.get("label") == "bad")
        print(f"Episodes: {total} | With outcome: {with_outcome} | "
              f"Good: {good} | Bad: {bad} | Unlabelled: {total-good-bad}")

    def review(self, n: int = 10):
        """Print a summary of the last n entries."""
        for i, e in enumerate(self.entries[-n:]):
            idx     = len(self.entries) - n + i
            pnl     = e.get("pnl")
            pnl_str = f"${pnl:+.2f}" if pnl is not None else "pending"
            label   = e.get("label") or "unlabelled"
            decs    = json.loads(e["completion"])
            actions = [f"{d['symbol']}:{d['action']}"
                       for d in decs if d.get("action") != "hold"][:5]
            print(f"[{idx:3d}] {label:>9} | P&L: {pnl_str:>10} | "
                  f"trades: {', '.join(actions) or 'all hold'}")
    def update_outcome(self, idx: int, equity_after: float):
        """
        Update the post-trade outcome for a logged episode.

        Args:
            idx          : index returned by log()
            equity_after : portfolio equity after trades

        Computes:
            pnl = equity_after - equity_before

        Notes:
        - If idx is None or invalid → silently ignore
        - Only updates once (prevents overwriting)
        """

        # --- Safety check ---
        if idx is None:
            return

        if not (0 <= idx < len(self.entries)):
            return

        entry = self.entries[idx]

        # --- Prevent double update ---
        if entry.get("equity_after") is not None:
            return

        entry["equity_after"] = equity_after
        entry["pnl"] = equity_after - entry["equity"]

        self._save_all()
    # ── My labelling rule: forward return with a no-trade band ────────────
    def label_by_forward_return(self, price_data: dict, horizon: int = 3,
                                band: float = 0.005, max_date: str = None):
        """
        Label each logged day by what happened to the traded stocks AFTER the
        decision, instead of next-day portfolio P&L:
          score of a buy  = return of the stock over the next `horizon` days
          score of a sell = minus that return (selling before a fall is right)
          right if score > band, wrong if score < -band, otherwise neutral
        Day label:
          good      : at least one right trade and no wrong trade
          corrected : some right, some wrong -> wrong trades are replaced by
                      HOLD in 'corrected_completion' (used for training)
          bad       : no right trade and at least one wrong trade
          neutral   : every trade inside the band (not used for training)
        Days whose horizon goes past max_date (end of the training period) or
        past the data stay unlabelled, so test-period prices never leak in.
        """
        dates = price_data["dates"]
        pos = {d: i for i, d in enumerate(dates)}
        last_i = pos.get(max_date, len(dates) - 1) if max_date else len(dates) - 1
        counts = {"good": 0, "corrected": 0, "bad": 0, "neutral": 0, "unlabelled": 0}
        for e in self.entries:
            d0 = e.get("date")
            if d0 not in pos or pos[d0] + horizon > last_i:
                e["label"] = None
                counts["unlabelled"] += 1
                continue
            d1 = dates[pos[d0] + horizon]
            decs = json.loads(e["completion"])
            right = wrong = 0
            corrected = []
            for d in decs:
                if d.get("action") not in ("buy", "sell"):
                    corrected.append(d)
                    continue
                p = price_data["prices"].get(d["symbol"], {})
                if d0 not in p or d1 not in p:
                    corrected.append(d)
                    continue
                r = p[d1] / p[d0] - 1
                score = r if d["action"] == "buy" else -r
                d["fwd_score"] = round(score, 4)
                if score > band:
                    right += 1
                    corrected.append(d)
                elif score < -band:
                    wrong += 1
                    corrected.append({**d, "action": "hold", "qty": None,
                                      "reason": f"corrected: {d['action']} was wrong ({score:+.2%})"})
                else:
                    corrected.append(d)
            if right and not wrong:
                e["label"] = "good"
            elif right and wrong:
                e["label"] = "corrected"
                e["corrected_completion"] = json.dumps(
                    [{k: x.get(k) for k in ("action", "symbol", "qty", "evidence", "reason")} for x in corrected])
            elif wrong:
                e["label"] = "bad"
            else:
                e["label"] = "neutral"
            e["fwd_right"], e["fwd_wrong"] = right, wrong
            counts[e["label"]] += 1
        self._save_all()
        print(f"Forward-return labels (horizon={horizon} days, band={band:.1%}): {counts}")
        return counts

    # ── Internal ───────────────────────────────────────────────────────────

    def _save_all(self):
        with open(self.save_path, "w", encoding="utf-8") as f:
            for e in self.entries:
                f.write(json.dumps(e, ensure_ascii=False) + "\n")


print("Training data utilities loaded.")


Training data utilities loaded.


### 1.14 Backtesting Infrastructure

> **Run this cell now, in order.** It only defines tools here; Task 4 uses them (4a collects training data, 4c compares the model before and after training).

Simulates N trading days using **historical prices** (yfinance) and **historical news**
(Alpaca, Drive-cached), with the agent loop making decisions each day.

#### Data flow per backtest day

```
fetch_historical_prices()           → daily closing prices (yfinance, Drive-cached)

For each trading day:
  1. llm_agent_decide(tok, model, state)
       Agent sees full universe + tool list, then:
       - Calls get_news(["NVDA","AAPL",...])  ← agent picks which stocks
         First call fetches from Alpaca & caches to Drive.
         Same-day re-runs hit the cache instantly (no API quota used).
       - Optionally calls get_rsi, get_macd, get_price_history
       - Outputs final JSON decision array

  2. SimulatedPortfolio.execute()   → simulate fills at that day's closing price

  3. TradingDataCollector.log()     → record conversation + decisions for SFT

  4. TradingDataCollector.update_outcome() → record next day's equity as outcome
```

#### Key classes

| Class | Purpose |
|-------|---------|
| `SimulatedPortfolio` | Tracks cash, positions, equity without any real orders |
| `fetch_historical_prices()` | Downloads/caches yfinance daily closes |
| `run_backtest()` | Orchestrates the full backtest loop |

#### After the backtest

```python
collector.auto_label()   # label good/bad based on PnL
collector.summary()      # print statistics
collector.review()       # inspect recent episodes
```
Then proceed to Task 4 to train the SFT model on the "good" episodes.


In [15]:
# =============================================================================
# Backtesting Infrastructure
# =============================================================================

class SimulatedPortfolio:
    """In-memory paper portfolio for backtesting — no real orders placed."""

    def __init__(self, initial_cash: float = 100_000.0):
        self.initial_cash = initial_cash
        self.cash         = initial_cash
        self.positions    = {}   # symbol -> {"qty": float, "avg_price": float}

    def equity(self, prices: dict) -> float:
        return self.cash + sum(
            p["qty"] * prices.get(sym, p["avg_price"])
            for sym, p in self.positions.items()
        )

    def get_state(self, prices: dict, timestamp: int = None) -> dict:
        pos = []
        for sym, p in self.positions.items():
            price = prices.get(sym, p["avg_price"])
            pos.append({
                "symbol":          sym,
                "qty":             p["qty"],
                "market_value":    round(p["qty"] * price, 2),
                "avg_entry_price": round(p["avg_price"], 2),
            })
        return {
            "cash":              round(self.cash, 2),
            "equity":            round(self.equity(prices), 2),
            "buying_power":      round(self.cash * 2, 2),
            "positions":         pos,
            "allowed_symbols":   WHITELIST,
            "max_qty_per_trade": MAX_QTY_PER_TRADE,
            "timestamp_unix":    timestamp or int(time.time()),
        }

    def execute(self, decision: dict, prices: dict):
        action = decision.get("action", "hold")
        symbol = decision.get("symbol")
        qty    = decision.get("qty")

        if action not in ("buy", "sell") or not symbol or not qty:
            return None
        if symbol not in prices:
            return None

        price = prices[symbol]

        if action == "buy":
            cost = qty * price
            if cost > self.cash:
                return None
            self.cash -= cost
            if symbol in self.positions:
                old       = self.positions[symbol]
                total_qty = old["qty"] + qty
                avg       = (old["qty"] * old["avg_price"] + cost) / total_qty
                self.positions[symbol] = {"qty": total_qty, "avg_price": avg}
            else:
                self.positions[symbol] = {"qty": float(qty), "avg_price": price}
            return {"status": "filled", "symbol": symbol, "side": "buy",
                    "qty": qty, "price": round(price, 2)}

        elif action == "sell":
            pos = self.positions.get(symbol)
            if pos is None or pos["qty"] < qty:
                return None
            self.cash += qty * price
            pos["qty"] -= qty
            if pos["qty"] <= 0:
                del self.positions[symbol]
            return {"status": "filled", "symbol": symbol, "side": "sell",
                    "qty": qty, "price": round(price, 2)}

        return None


def fetch_historical_prices(symbols: list = None, period: str = "6mo") -> dict:
    """
    Download historical daily closing prices via yfinance (Drive-cached daily).
    Returns {"dates": [...], "prices": {sym: {date: price}}, "timestamp": ...}
    """
    if symbols is None:
        symbols = WHITELIST

    cache_path = os.path.join(CACHE_DIR, "historical_prices.json")
    if os.path.exists(cache_path):
        with open(cache_path) as f:
            cached = json.load(f)
        if time.time() - cached.get("timestamp", 0) < 86400:
            print(f"Loaded cached price data ({len(cached['dates'])} trading days)")
            return cached

    print(f"Downloading {period} of daily prices for {len(symbols)} symbols...")
    df = yf.download(symbols, period=period, progress=False)

    close_df = df["Close"] if isinstance(df.columns, pd.MultiIndex) else df[["Close"]]
    if not isinstance(df.columns, pd.MultiIndex):
        close_df.columns = [symbols[0]]

    dates  = [d.strftime("%Y-%m-%d") for d in close_df.index]
    prices = {}
    for sym in symbols:
        if sym in close_df.columns:
            sym_prices = {}
            for d, v in zip(dates, close_df[sym]):
                try:
                    fv = float(v)
                    if fv == fv:   # NaN check
                        sym_prices[d] = round(fv, 2)
                except (ValueError, TypeError):
                    pass
            prices[sym] = sym_prices

    result = {"dates": dates, "prices": prices, "timestamp": time.time()}
    with open(cache_path, "w") as f:
        json.dump(result, f)
    print(f"Cached {len(dates)} trading days to Drive")
    return result


def run_backtest(tok, model, collector=None, n_days: int = 20,
                 initial_cash: float = 100_000.0, verbose: bool = True,
                 skip_last: int = 0) -> dict:
    """
    Replay N historical trading days using the agent decision loop.

    For each day:
      1. llm_agent_decide()  — agent calls tools (incl. get_news) on demand
      2. Execute simulated trades at that day's closing prices
      3. Log episode to collector (if provided)

    Args:
        tok          : tokenizer
        model        : base model (or LoRA fine-tuned model for comparison)
        collector    : TradingDataCollector instance (or None to skip logging)
        n_days       : number of most-recent trading days to replay
        initial_cash : starting portfolio cash
        verbose      : print per-day progress

    Returns:
        dict with equity_history, trades, total_return, final_equity
    """
    price_data = fetch_historical_prices()
    all_dates  = price_data["dates"]
    # [CHANGE] skip_last: leave out the most recent days (held-out test period)
    end_i      = len(all_dates) - skip_last
    n_days     = min(n_days, end_i)
    dates      = all_dates[end_i - n_days:end_i]
    reset_agent_stats()

    portfolio      = SimulatedPortfolio(initial_cash)
    equity_history = []
    all_trades     = []
    entry_indices  = []

    if verbose:
        print(f"\n{'='*60}")
        print(f"BACKTEST  {dates[0]}  →  {dates[-1]}  ({len(dates)} days)")
        print(f"Initial cash: ${initial_cash:,.0f}  |  Universe: {len(WHITELIST)} stocks")
        print(f"Agent: up to 6 tool-call turns per day")
        print(f"{'='*60}")

    for date in dates:
        day_prices = {
            sym: price_data["prices"].get(sym, {}).get(date)
            for sym in WHITELIST
        }
        day_prices = {k: v for k, v in day_prices.items() if v is not None}
        if len(day_prices) < 5:
            continue

        print(f"{date}:")
        # Agent decides; get_news tool calls are cached by date
        try:
            ts = int(datetime.datetime.strptime(date, "%Y-%m-%d").timestamp())
        except Exception:
            ts = int(time.time())

        state         = portfolio.get_state(day_prices, timestamp=ts)
        equity_before = state["equity"]

        decisions, sft_prompt, sft_completion, tool_calls_made = llm_agent_decide(
            tok, model, state, universe=WHITELIST, date=date
        )

        # Simulate trades
        day_trades = 0
        for d in decisions:
            result = portfolio.execute(d, day_prices)
            if result:
                # [CHANGE] simple transaction cost on every fill
                portfolio.cash -= result["qty"] * result["price"] * TRANSACTION_COST_BPS / 10_000
                if verbose:
                    print(f"    filled: {result['side']} {result['qty']} {result['symbol']} @ {result['price']}")
                all_trades.append({**result, "date": date})
                day_trades += 1

        equity_after = portfolio.equity(day_prices)
        equity_history.append({"date": date, "equity": round(equity_after, 2)})

        # Log episode
        if collector is not None:
            idx = collector.log(sft_prompt, sft_completion, decisions, equity_before, tool_calls_made)
            if idx is not None:                       # [CHANGE] remember the day for labelling
                collector.entries[idx]["date"] = date
                collector._save_all()
            entry_indices.append(idx)
            if len(entry_indices) > 1:
                collector.update_outcome(entry_indices[-2], equity_after)

        pnl = equity_after - initial_cash
        if verbose:
            print(f"  Equity: ${equity_after:>10,.2f}  |  "
                  f"P&L: ${pnl:>+9,.2f}  |  Trades: {day_trades}")

    # Update last entry outcome
    if collector and entry_indices:
        final_eq = equity_history[-1]["equity"] if equity_history else initial_cash
        collector.update_outcome(entry_indices[-1], final_eq)

    final_equity = equity_history[-1]["equity"] if equity_history else initial_cash
    total_return = (final_equity - initial_cash) / initial_cash * 100

    # [CHANGE] benchmarks over the same days: buy-and-hold SPY, equal-weight FOCUS_LIST
    def _bh(sym):
        p = price_data["prices"].get(sym, {})
        return (p[dates[-1]] / p[dates[0]] - 1) * 100 if dates[0] in p and dates[-1] in p else float("nan")
    spy_return = _bh("SPY")
    ew = [_bh(s) for s in FOCUS_LIST]
    ew = [x for x in ew if x == x]
    ew_return = sum(ew) / len(ew) if ew else float("nan")
    trade_days = len({t["date"] for t in all_trades})

    if verbose:
        print(f"\n{'='*60}")
        print(f"RESULTS")
        print(f"  Period:  {dates[0]}  →  {dates[-1]}")
        print(f"  Initial: ${initial_cash:>12,.2f}")
        print(f"  Final:   ${final_equity:>12,.2f}")
        print(f"  Return:  {total_return:>+11.2f}%")
        print(f"  Trades:  {len(all_trades):>12d}  (on {trade_days} of {len(dates)} days)")
        print(f"  Buy-and-hold SPY:          {spy_return:>+8.2f}%")
        print(f"  Equal-weight focus list:   {ew_return:>+8.2f}%")
        print(f"  Agent stats: {AGENT_STATS}")
        print(f"{'='*60}")

    return {
        "equity_history": equity_history,
        "trades":         all_trades,
        "total_return":   total_return,
        "final_equity":   final_equity,
        "dates":          dates,
        "spy_return":     spy_return,
        "ew_return":      ew_return,
        "trade_days":     trade_days,
        "agent_stats":    dict(AGENT_STATS),
    }


print("Backtesting infrastructure loaded.")


Backtesting infrastructure loaded.


### 1.15 Test Agent Decision Engine

Runs a quick smoke test of `llm_agent_decide()` using only the tools that are
currently implemented (i.e. registered with a non-None `func`).

**At this point (before Task 2):**
- `get_news` is available — the agent can fetch financial news
- `get_price_history`, `get_rsi`, `get_macd`, `get_global_quote` are not yet available

After completing Task 2 and registering your tools, re-run this cell to see
the agent use all of your implementations.


In [23]:
# Quick smoke test — only tools implemented so far will be available
print("Available tools before Task 2:")
list_tools()

print("\nRunning agent on 5-symbol subset...")
state = get_state(tc)
decisions, sft_prompt, sft_completion, tool_calls_made = llm_agent_decide(
    tok, model, state, universe=WHITELIST[:5]
)

print(f"\nDecisions ({len(decisions)}):")
for d in decisions:
    sym    = d.get("symbol", "?")
    act    = d["action"].upper()
    qty    = d.get("qty", "-")
    reason = d.get("reason", "")[:60]
    print(f"  [{sym}] {act:5s}  qty={str(qty):>4}  {reason}")

print(f"\nSFT prompt  : {len(sft_prompt):,} chars")
print(f"SFT completion: {len(sft_completion):,} chars")


Available tools before Task 2:
Tool                   Status         Description
----------------------------------------------------------------------
  get_news             OK             Get recent news headlines (with publish time) for specific stocks. Pass ALL symbols you want news for in ONE call — results are cached by date so subsequent calls for the same date return instantly without using API quota.
  get_price_history    OK             Recent daily OHLCV prices for a stock (yfinance)
  get_rsi              OK             RSI momentum indicator (>70 overbought, <30 oversold) via Alpha Vantage
  get_macd             OK             MACD trend indicator computed from daily closes (bullish/bearish crossover)
  get_global_quote     OK             Latest price quote with daily change via Alpha Vantage
  get_company_research OK             SEC EDGAR evidence: recent 8-K/10-Q/10-K filings and quarterly EPS/revenue vs last year
  get_market_context   OK             Overall market regi

### Task 1 Complete!

You now have all core infrastructure running:
- LLM model loaded
- Alpaca trading client connected
- News cached to Google Drive
- Decision engine ready
- Training data collector ready (for Task 4)
- Backtesting engine ready (for Task 4)

**Next: Implement Tasks 2-5 below.**

---

## Task 2: [TODO] Implement MCP Tool Functions

### Objective

Implement the **data retrieval tools** that the trading agent uses to research stocks
before making decisions. When the agent outputs a tool call, `dispatch_tool()` routes
it to your function — so the quality of your implementations directly determines
how well-informed the agent's decisions are.

### Tool Registry Reference

All tool functions you implement in this task must be registered using the `register_tool()`
function from **Section 1.10**. After you write each function, call:

```python
register_tool(
    name        = "your_tool_name",
    description = "brief description for the agent",
    params      = {"param": "type — description"},
    func        = your_function,
)
```

See Section 1.10 for the complete Tool Registry API documentation.

### How a tool call reaches your code

```
Agent outputs:
  <tool_call>{"name": "get_rsi", "arguments": {"symbol": "NVDA", "time_period": 14}}</tool_call>

dispatch_tool("get_rsi", {"symbol": "NVDA", "time_period": 14})
  └─ _tool_registry["get_rsi"]["func"]("NVDA", time_period=14)
       └─ YOUR get_rsi("NVDA", time_period=14)   ← implement this
            └─ returns: "RSI(14) for NVDA: 62.3 (neutral) [2024-01-15]"
```

### [TODO] Required: implement and register these tools

**Implementation order:** Start with `get_news_for_symbols` (Section 1.9) as it demonstrates the complete pattern: API call → error handling → caching → formatting. Then implement the remaining functions following the same pattern.

| Function | Data source | What to implement |
|----------|------------|-------------------|
| `get_news_for_symbols(symbols, limit, date)` | **Alpaca** | News with caching |
| `get_price_history(symbol, period)` | **yfinance** | Provided: recent OHLCV table |
| `get_rsi(symbol, time_period)` | **Alpha Vantage** | RSI value + signal |
| `get_macd(symbol)` | **yfinance** | MACD, signal, histogram (computed from daily closes) |
| `get_global_quote(symbol)` | **Alpha Vantage** | Live price, change, volume |

> Alpha Vantage made `NEWS_SENTIMENT` and `MACD` premium-only, so news comes from Alpaca (Section 1.9)
> and MACD is computed from yfinance prices. RSI and GLOBAL_QUOTE still work with the free Alpha Vantage key.

After implementing each function you **must** call `register_tool()` to update the
registry — otherwise the agent won't know the tool is ready.

### Alpha Vantage quick reference

```python
# RSI
resp = requests.get(ALPHA_VANTAGE_BASE_URL, params={
    "function": "RSI", "symbol": symbol,
    "interval": "daily", "time_period": time_period,
    "series_type": "close", "apikey": ALPHA_VANTAGE_API_KEY,
})
data = resp.json()
# data["Technical Analysis: RSI"] -> {date: {"RSI": "65.32"}, ...}

# Global Quote
resp = requests.get(ALPHA_VANTAGE_BASE_URL, params={
    "function": "GLOBAL_QUOTE", "symbol": symbol,
    "apikey": ALPHA_VANTAGE_API_KEY,
})
# data["Global Quote"] -> {"05. price": "182.63", "09. change": "+1.20", ...}
```

### yfinance quick reference

```python
import yfinance as yf
ticker = yf.Ticker("AAPL")
hist   = ticker.history(period="5d")   # pandas DataFrame
# columns: Open, High, Low, Close, Volume
# index:   DatetimeIndex

# MACD from daily closes (Alpha Vantage's MACD is premium-only)
close  = ticker.history(period="6mo")["Close"]
macd   = close.ewm(span=12, adjust=False).mean() - close.ewm(span=26, adjust=False).mean()
signal = macd.ewm(span=9, adjust=False).mean()
hist   = macd - signal
```

### Optional: add your own custom tool

Implement any additional data source (earnings, options flow, sector ETF performance,
macroeconomic indicators, etc.) and register it so the agent can call it automatically:

```python
def get_earnings(symbol: str) -> str:
    """Fetch latest quarterly earnings from Alpha Vantage."""
    # function="EARNINGS" -> data["quarterlyEarnings"][0]
    ...

register_tool(
    name        = "get_earnings",
    description = "Get quarterly earnings data for a stock",
    params      = {"symbol": "str — stock ticker"},
    func        = get_earnings,
)
# Agent now calls get_earnings automatically when it wants earnings data
```

### Grading criteria

- [ ] `get_news_for_symbols` implements full caching + API call pattern. *Evidence: Test 1.9 (the repeated query prints `Cache hit`).*
- [ ] At least **2** Alpha Vantage functions implemented; each handles API errors gracefully (try/except); all registered with `register_tool(..., func=your_fn)`. *Evidence: Test Task 2 output and `list_tools()` in 1.15.*
- `get_price_history` is provided. Optional custom tools are welcome; describe them in your report.


In [16]:
# =============================================================================
# Task 2 — MCP Tool Functions
#   Required : get_price_history, get_rsi, get_macd, get_global_quote,
#              get_extra_context
#   My tools : get_company_research (SEC EDGAR evidence cards),
#              get_market_context (SPY trend + VIX risk regime)
#
# Design rules used by every tool:
#   * date=None  -> live trading (use the latest data)
#     date="YYYY-MM-DD" -> backtest: ONLY data dated on/before that day is used,
#     so the agent cannot see the future (no look-ahead).
#   * Every fact the agent may cite carries an evidence ID in square brackets,
#     e.g. [P-AAPL], [R-AAPL], [S-AAPL-1], [M-SPY]. Section 1.12 checks that a
#     trade cites IDs that really appeared in its tool results.
#   * Every tool catches its own errors and returns a short message instead of
#     crashing the agent loop.
# =============================================================================
import inspect
import numpy as np

# -----------------------------------------------------------------------------
# Shared helper: daily price history from yfinance (one download per symbol
# per day, kept in memory and on disk)
# -----------------------------------------------------------------------------
_PRICE_MEM = {}
LIVE_PRICE_MAX_AGE = 30 * 60   # live trading: refresh prices every 30 minutes

def _daily_history(symbol: str, max_age: float = None) -> pd.DataFrame:
    """
    ~1 year of daily OHLCV for symbol (index = date), cached in memory and on disk.
    max_age=None  -> any copy downloaded today is fine (backtest)
    max_age=1800  -> re-download if the copy is older than 30 minutes (live)
    """
    today = datetime.date.today().isoformat()
    mem = _PRICE_MEM.get(symbol)
    if mem and mem[2] == today and (max_age is None or time.time() - mem[0] < max_age):
        return mem[1]
    path = os.path.join(CACHE_DIR, "prices", f"{symbol.replace('^', '_')}_{today}.csv")
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if os.path.exists(path) and (max_age is None or time.time() - os.path.getmtime(path) < max_age):
        df = pd.read_csv(path, index_col=0, parse_dates=True)
    else:
        df = yf.Ticker(symbol).history(period="1y", auto_adjust=False)
        if df.empty:
            raise ValueError(f"no price data for {symbol}")
        df.index = pd.to_datetime(df.index).tz_localize(None).normalize()
        df = df[["Open", "High", "Low", "Close", "Volume"]]
        df.to_csv(path)
    _PRICE_MEM[symbol] = (time.time(), df, today)
    return df


def _history_until(symbol: str, date: str = None) -> pd.DataFrame:
    """Daily history up to and including `date` (latest data when date is None)."""
    if date is None:
        df = _daily_history(symbol, max_age=LIVE_PRICE_MAX_AGE)
    else:
        df = _daily_history(symbol)
        df = df[df.index <= pd.Timestamp(date)]
    if df.empty:
        raise ValueError(f"no price data for {symbol} on/before {date}")
    return df


# -----------------------------------------------------------------------------
# Alpha Vantage helper with a daily call budget (free tier = 25 calls/day)
# -----------------------------------------------------------------------------
AV_DAILY_BUDGET = 22          # keep a few calls spare

def _av_budget_path():
    return os.path.join(CACHE_DIR, f"av_calls_{datetime.date.today().isoformat()}.json")

def _av_calls_used() -> int:
    try:
        with open(_av_budget_path(), encoding="utf-8") as f:
            return json.load(f)["calls"]
    except Exception:
        return 0

def _av_request(params: dict) -> dict:
    """Call Alpha Vantage. Raises RuntimeError on budget/limit/API errors."""
    if _av_calls_used() >= AV_DAILY_BUDGET:
        raise RuntimeError("Alpha Vantage daily budget used up")
    resp = requests.get(ALPHA_VANTAGE_BASE_URL,
                        params={**params, "apikey": ALPHA_VANTAGE_API_KEY},
                        timeout=30)
    with open(_av_budget_path(), "w", encoding="utf-8") as f:
        json.dump({"calls": _av_calls_used() + 1}, f)
    if resp.status_code != 200:
        raise RuntimeError(f"Alpha Vantage HTTP {resp.status_code}")
    data = resp.json()
    for k in ("Error Message", "Information", "Note"):
        if k in data:   # invalid symbol, rate limit or premium-only message
            raise RuntimeError(f"Alpha Vantage: {str(data[k])[:120]}")
    return data


def _rsi_from_closes(close: pd.Series, n: int = 14) -> pd.Series:
    """Wilder RSI computed locally (fallback when Alpha Vantage is unavailable)."""
    delta = close.diff()
    gain = delta.clip(lower=0).ewm(alpha=1 / n, adjust=False).mean()
    loss = (-delta.clip(upper=0)).ewm(alpha=1 / n, adjust=False).mean()
    rs = gain / loss.replace(0, np.nan)
    return (100 - 100 / (1 + rs)).fillna(100.0)


def _rsi_label(v: float) -> str:
    if v >= 70: return "overbought"
    if v <= 30: return "oversold"
    if v >= 60: return "neutral, approaching overbought"
    if v <= 40: return "neutral, approaching oversold"
    return "neutral"


# ── Required: get_price_history (provided, extended with date + evidence ID) ─

_PERIOD_ROWS = {"5d": 5, "10d": 10, "1mo": 21, "3mo": 63}

def get_price_history(symbol: str, period: str = "5d", date: str = None) -> str:
    try:
        symbol = symbol.upper().strip()
        rows = _PERIOD_ROWS.get(str(period), 5)
        hist = _history_until(symbol, date).tail(rows)
        lines = [f"[P-{symbol}] Price history for {symbol} ({period}, yfinance):"]
        for d, row in hist.iterrows():
            lines.append(f"  {d.strftime('%Y-%m-%d')}: O={row['Open']:.2f} H={row['High']:.2f} "
                         f"L={row['Low']:.2f} C={row['Close']:.2f} V={int(row['Volume'])}")
        if len(hist) >= 2:
            chg = (hist["Close"].iloc[-1] / hist["Close"].iloc[0] - 1) * 100
            lines.append(f"  Change over period: {chg:+.2f}%")
        return "\n".join(lines)
    except Exception as e:
        return f"Price history fetch failed for {symbol}: {e}"


# ── Required: get_rsi (Alpha Vantage, cached, local fallback) ────────────────

def get_rsi(symbol: str, time_period: int = 14, date: str = None, interval: str = "daily") -> str:
    """
    RSI from Alpha Vantage (daily). One API call returns the whole RSI history,
    so it is cached per symbol and reused for backtest dates.
    If Alpha Vantage fails (limit reached, network), RSI is computed from
    yfinance closes instead, and the output says so.
    """
    symbol = symbol.upper().strip()
    try:
        time_period = int(time_period)
    except Exception:
        time_period = 14
    target = date or datetime.date.today().isoformat()
    cache_path = os.path.join(CACHE_DIR, f"av_rsi_{symbol}_{time_period}.json")

    series = None
    # 1) cached Alpha Vantage series, if it covers the date we need
    try:
        if os.path.exists(cache_path):
            with open(cache_path, encoding="utf-8") as f:
                cached = json.load(f)
            if cached["fetched"] >= target:
                series = cached["series"]
    except Exception:
        series = None
    # 2) fresh Alpha Vantage call
    if series is None:
        try:
            data = _av_request({"function": "RSI", "symbol": symbol, "interval": "daily",
                                "time_period": time_period, "series_type": "close"})
            raw = data["Technical Analysis: RSI"]
            series = {d: float(v["RSI"]) for d, v in raw.items()}
            with open(cache_path, "w", encoding="utf-8") as f:
                json.dump({"fetched": datetime.date.today().isoformat(), "series": series}, f)
        except Exception:
            series = None    # fall back to local computation below
    # 3) use Alpha Vantage series (latest date on/before target)
    if series:
        valid = sorted(d for d in series if d <= target)
        if valid:
            d = valid[-1]
            v = series[d]
            return f"[R-{symbol}] RSI({time_period}) for {symbol}: {v:.1f} ({_rsi_label(v)}) [{d}, Alpha Vantage]"
    # 4) fallback: compute locally from yfinance
    try:
        close = _history_until(symbol, date)["Close"]
        rsi = _rsi_from_closes(close, time_period)
        v, d = float(rsi.iloc[-1]), rsi.index[-1].strftime("%Y-%m-%d")
        return f"[R-{symbol}] RSI({time_period}) for {symbol}: {v:.1f} ({_rsi_label(v)}) [{d}, computed from yfinance]"
    except Exception as e:
        return f"RSI unavailable for {symbol}: {e}"


# ── Required: get_macd (computed from yfinance closes) ───────────────────────

def get_macd(symbol: str, date: str = None, interval: str = "daily") -> str:
    """MACD = EMA12 - EMA26, Signal = EMA9(MACD), Hist = MACD - Signal."""
    symbol = symbol.upper().strip()
    try:
        close = _history_until(symbol, date)["Close"]
        if len(close) < 35:
            return f"MACD unavailable for {symbol}: not enough price history"
        macd = close.ewm(span=12, adjust=False).mean() - close.ewm(span=26, adjust=False).mean()
        signal = macd.ewm(span=9, adjust=False).mean()
        hist = macd - signal
        m, s, h, h_prev = macd.iloc[-1], signal.iloc[-1], hist.iloc[-1], hist.iloc[-2]
        trend = "bullish" if m > s else "bearish"
        if h > 0 and h_prev <= 0:
            trend += ", fresh bullish crossover"
        elif h < 0 and h_prev >= 0:
            trend += ", fresh bearish crossover"
        d = close.index[-1].strftime("%Y-%m-%d")
        return (f"[D-{symbol}] MACD for {symbol} [{d}]: MACD={m:.4f}, Signal={s:.4f}, "
                f"Hist={h:.4f} ({trend})")
    except Exception as e:
        return f"MACD unavailable for {symbol}: {e}"


# ── Required: get_global_quote (Alpha Vantage, cached, fallback) ─────────────

QUOTE_MAX_AGE = 15 * 60    # reuse a live quote for 15 minutes (saves API calls)
_QUOTE_MEM = {}

def get_global_quote(symbol: str, date: str = None) -> str:
    """
    Live: Alpha Vantage GLOBAL_QUOTE (cached 15 min; yfinance if AV fails).
    Backtest (date given): that day's close from yfinance — a real-time quote
    for a past date does not exist, and calling AV would leak today's price.
    """
    symbol = symbol.upper().strip()
    try:
        if date is not None:
            df = _history_until(symbol, date)
            last, prev = df.iloc[-1], (df.iloc[-2] if len(df) > 1 else df.iloc[-1])
            chg = last["Close"] - prev["Close"]
            pct = chg / prev["Close"] * 100
            return (f"[Q-{symbol}] Quote {symbol} [{df.index[-1].strftime('%Y-%m-%d')} close]: "
                    f"${last['Close']:.2f} | change: {chg:+.2f} ({pct:+.2f}%) | vol: {int(last['Volume']):,}")

        cached = _QUOTE_MEM.get(symbol)
        if cached and time.time() - cached[0] < QUOTE_MAX_AGE:
            return cached[1]
        try:
            q = _av_request({"function": "GLOBAL_QUOTE", "symbol": symbol})["Global Quote"]
            text = (f"[Q-{symbol}] Quote {symbol}: ${float(q['05. price']):.2f} | change: "
                    f"{float(q['09. change']):+.2f} ({q['10. change percent']}) | "
                    f"vol: {int(q['06. volume']):,} [{q['07. latest trading day']}, Alpha Vantage]")
        except Exception as av_err:
            fi = yf.Ticker(symbol).fast_info
            price, prev = float(fi["last_price"]), float(fi["previous_close"])
            text = (f"[Q-{symbol}] Quote {symbol}: ${price:.2f} | change: {price - prev:+.2f} "
                    f"({(price / prev - 1) * 100:+.2f}%) [yfinance; Alpha Vantage unavailable]")
        _QUOTE_MEM[symbol] = (time.time(), text)
        return text
    except Exception as e:
        return f"Quote unavailable for {symbol}: {e}"


# =============================================================================
# My tool 1: get_company_research — SEC EDGAR evidence cards
# =============================================================================
# EDGAR is free but SEC requires a User-Agent with your name and email.
# Add this line to the .env file:  SEC_USER_AGENT=Your Name your@email.com
from dotenv import load_dotenv
load_dotenv()
SEC_USER_AGENT = os.getenv("SEC_USER_AGENT", "")

SEC_ITEM_NAMES = {
    "1.01": "entered a material agreement", "1.02": "terminated a material agreement",
    "1.05": "material cybersecurity incident", "2.01": "completed an acquisition or sale of assets",
    "2.02": "results of operations (earnings release)", "2.03": "new financial obligation (debt)",
    "2.05": "restructuring / exit costs", "2.06": "material impairment",
    "3.01": "delisting notice", "4.02": "prior financials should no longer be relied on",
    "5.02": "director or officer change", "5.07": "shareholder vote results",
    "7.01": "Regulation FD disclosure", "8.01": "other material event",
}
SEC_FORMS = {"8-K", "10-Q", "10-K"}
_SEC_MEM = {}

def _sec_get_json(url: str, cache_name: str, max_age_days: int = 1):
    """GET a JSON file from SEC with on-disk caching."""
    path = os.path.join(CACHE_DIR, "sec", cache_name)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if os.path.exists(path) and time.time() - os.path.getmtime(path) < max_age_days * 86400:
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    if not SEC_USER_AGENT:
        raise RuntimeError("SEC_USER_AGENT missing in .env")
    time.sleep(0.15)                     # SEC allows max 10 requests/second
    resp = requests.get(url, headers={"User-Agent": SEC_USER_AGENT}, timeout=30)
    if resp.status_code != 200:
        raise RuntimeError(f"SEC HTTP {resp.status_code}")
    data = resp.json()
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f)
    return data


def _sec_cik(symbol: str):
    """Ticker -> 10-digit CIK, or None for ETFs / unknown tickers."""
    if "tickers" not in _SEC_MEM:
        raw = _sec_get_json("https://www.sec.gov/files/company_tickers.json",
                            "company_tickers.json", max_age_days=7)
        _SEC_MEM["tickers"] = {v["ticker"].upper(): str(v["cik_str"]).zfill(10)
                               for v in raw.values()}
    return _SEC_MEM["tickers"].get(symbol.upper().replace(".", "-"))


def _sec_quarterly_fact(cik: str, concepts: list, before: str):
    """
    Latest 3-month value of an XBRL concept filed before `before`, plus the
    same quarter one year earlier. Returns dict or None.
    """
    for concept in concepts:
        try:
            data = _sec_get_json(
                f"https://data.sec.gov/api/xbrl/companyconcept/CIK{cik}/us-gaap/{concept}.json",
                f"concept_{cik}_{concept}.json")
        except Exception:
            continue
        facts = []
        for unit_vals in data.get("units", {}).values():
            for f in unit_vals:
                if not f.get("start") or f.get("form") not in ("10-Q", "10-K"):
                    continue
                days = (pd.Timestamp(f["end"]) - pd.Timestamp(f["start"])).days
                if 80 <= days <= 100 and f.get("filed", "9999") < before:
                    facts.append(f)
        if not facts:
            continue
        latest = max(facts, key=lambda f: (f["end"], f["filed"]))
        target_end = pd.Timestamp(latest["end"]) - pd.Timedelta(days=365)
        prior = [f for f in facts if abs((pd.Timestamp(f["end"]) - target_end).days) <= 15]
        prior = max(prior, key=lambda f: f["filed"]) if prior else None
        return {"concept": concept, "latest": latest, "prior": prior}
    return None


def get_company_research(symbol: str, date: str = None) -> str:
    """
    Evidence card for one company from SEC EDGAR:
      * recent 8-K / 10-Q / 10-K filings (last 60 days) with item meanings
      * latest quarterly diluted EPS and revenue vs the same quarter last year
    Only filings dated strictly BEFORE `date` are used in a backtest.
    """
    symbol = symbol.upper().strip()
    asof = date or (datetime.date.today() + datetime.timedelta(days=1)).isoformat()
    try:
        cik = _sec_cik(symbol)
        if cik is None:
            return f"No SEC filings for {symbol} (ETF or not an SEC filer)."

        sub = _sec_get_json(f"https://data.sec.gov/submissions/CIK{cik}.json",
                            f"submissions_{cik}.json")
        rec = sub["filings"]["recent"]
        cutoff = (pd.Timestamp(asof) - pd.Timedelta(days=60)).strftime("%Y-%m-%d")
        filings = []
        for form, fdate, items in zip(rec["form"], rec["filingDate"], rec.get("items", [""] * len(rec["form"]))):
            if form in SEC_FORMS and cutoff <= fdate < asof:
                filings.append((fdate, form, items or ""))
        filings.sort(reverse=True)

        lines = [f"=== SEC research: {symbol} ({sub.get('name', '')}) as of {date or 'today'} ==="]
        if not filings:
            lines.append("  No 8-K/10-Q/10-K filings in the last 60 days.")
        for k, (fdate, form, items) in enumerate(filings[:5], 1):
            meaning = ""
            if form == "8-K" and items:
                codes = [c.strip() for c in items.split(",") if c.strip() and c.strip() != "9.01"]
                meaning = "; ".join(SEC_ITEM_NAMES.get(c, f"item {c}") for c in codes)
            elif form == "10-Q":
                meaning = "quarterly report"
            elif form == "10-K":
                meaning = "annual report"
            lines.append(f"  [S-{symbol}-{k}] {fdate} | SEC {form} | {meaning}")

        for label, concepts, tag in [
            ("Diluted EPS", ["EarningsPerShareDiluted"], "EPS"),
            ("Revenue", ["RevenueFromContractWithCustomerExcludingAssessedTax", "Revenues",
                         "SalesRevenueNet"], "REV"),
        ]:
            fact = _sec_quarterly_fact(cik, concepts, asof)
            if not fact:
                continue
            L, P = fact["latest"], fact["prior"]
            val = L["val"]
            txt = f"{val:,.2f}" if tag == "EPS" else f"${val / 1e9:,.2f}B"
            if P and P["val"]:
                yoy = (val - P["val"]) / abs(P["val"]) * 100
                txt += f" vs {P['val']:,.2f} a year earlier ({yoy:+.1f}% YoY)" if tag == "EPS" \
                    else f" vs ${P['val'] / 1e9:,.2f}B a year earlier ({yoy:+.1f}% YoY)"
            lines.append(f"  [F-{symbol}-{tag}] {L['filed']} | SEC {L['form']} | "
                         f"{label} for quarter ending {L['end']}: {txt}")
        return "\n".join(lines)
    except Exception as e:
        return f"SEC research unavailable for {symbol}: {e}"


# =============================================================================
# My tool 2: get_market_context — SPY trend + VIX (risk-on / risk-off)
# =============================================================================

def get_market_context(date: str = None) -> str:
    """Overall market regime from SPY moving averages and the VIX level."""
    try:
        spy = _history_until("SPY", date)["Close"]
        d = spy.index[-1].strftime("%Y-%m-%d")
        last, ma20, ma50 = spy.iloc[-1], spy.tail(20).mean(), spy.tail(50).mean()
        ret5 = (spy.iloc[-1] / spy.iloc[-6] - 1) * 100 if len(spy) > 5 else float("nan")
        trend = ("uptrend" if last > ma20 > ma50 else
                 "downtrend" if last < ma20 < ma50 else "mixed")
        lines = [f"[M-SPY] {d} | SPY {last:.2f}, 20-day avg {ma20:.2f}, 50-day avg {ma50:.2f}, "
                 f"5-day return {ret5:+.2f}% -> {trend}"]
        try:
            vix = float(_history_until("^VIX", date)["Close"].iloc[-1])
            vix_note = "calm" if vix < 15 else "normal" if vix < 20 else "elevated" if vix < 30 else "fearful"
            lines.append(f"[M-VIX] {d} | VIX {vix:.1f} ({vix_note})")
        except Exception:
            vix = None
        risk_off = trend == "downtrend" or (vix is not None and vix >= 25)
        lines.append(f"Market regime: {'RISK-OFF (prefer smaller positions)' if risk_off else 'RISK-ON'}")
        return "\n".join(lines)
    except Exception as e:
        return f"Market context unavailable: {e}"


# =============================================================================
# Research trigger — decides WHEN deeper research is worth it (used in 1.12)
# =============================================================================

def research_flags(symbol: str, date: str = None) -> list:
    """
    Reasons to research a stock before trading it:
      * large move: last daily return beyond 2 standard deviations (20 days)
      * earnings just released: 8-K item 2.02 filed in the last 5 days
    Returns a list of short reason strings (empty = no extra research needed).
    """
    reasons = []
    try:
        close = _history_until(symbol, date)["Close"]
        rets = close.pct_change().dropna()
        if len(rets) > 21:
            last, sd = rets.iloc[-1], rets.iloc[-21:-1].std()
            if sd > 0 and abs(last) > 2 * sd:
                reasons.append(f"large move {last * 100:+.1f}% (>2 std)")
    except Exception:
        pass
    try:
        cik = _sec_cik(symbol)
        if cik:
            asof = date or (datetime.date.today() + datetime.timedelta(days=1)).isoformat()
            since = (pd.Timestamp(asof) - pd.Timedelta(days=5)).strftime("%Y-%m-%d")
            sub = _sec_get_json(f"https://data.sec.gov/submissions/CIK{cik}.json",
                                f"submissions_{cik}.json")
            rec = sub["filings"]["recent"]
            for form, fdate, items in zip(rec["form"], rec["filingDate"], rec.get("items", [])):
                if form == "8-K" and since <= fdate < asof and "2.02" in (items or ""):
                    reasons.append(f"earnings released {fdate}")
                    break
    except Exception:
        pass
    return reasons


# =============================================================================
# Register all tools (overwrites the func=None placeholders from 1.10)
# =============================================================================
register_tool("get_price_history",
              "Recent daily OHLCV prices for a stock (yfinance)",
              {"symbol": "str", "period": 'str — "5d", "1mo", "3mo"'},
              func=get_price_history)
register_tool("get_rsi",
              "RSI momentum indicator (>70 overbought, <30 oversold) via Alpha Vantage",
              {"symbol": "str", "time_period": "int (default 14)"},
              func=get_rsi)
register_tool("get_macd",
              "MACD trend indicator computed from daily closes (bullish/bearish crossover)",
              {"symbol": "str"},
              func=get_macd)
register_tool("get_global_quote",
              "Latest price quote with daily change via Alpha Vantage",
              {"symbol": "str"},
              func=get_global_quote)
register_tool("get_company_research",
              "SEC EDGAR evidence: recent 8-K/10-Q/10-K filings and quarterly EPS/revenue vs last year",
              {"symbol": "str"},
              func=get_company_research)
register_tool("get_market_context",
              "Overall market regime: SPY trend and VIX fear level (risk-on / risk-off)",
              {},
              func=get_market_context)


# =============================================================================
# get_extra_context — bundle of tool outputs passed to the agent in Task 5
# =============================================================================

def get_extra_context(symbols: list, date: str = None, include_research: bool = True) -> str:
    """
    Run the data tools for a few symbols and join the results.
    Each call is wrapped in try/except so one failure does not block the rest.
    """
    if isinstance(symbols, str):
        symbols = [symbols]
    parts = []
    try:
        parts.append(get_market_context(date=date))
    except Exception as e:
        parts.append(f"Market context failed: {e}")
    for symbol in symbols:
        block = [f"=== Extra Context for {symbol} ==="]
        tools = [
            lambda s: get_price_history(s, "5d", date=date),
            lambda s: get_rsi(s, date=date),
            lambda s: get_macd(s, date=date),
            lambda s: get_global_quote(s, date=date),
        ]
        for tool in tools:
            try:
                block.append(tool(symbol))
            except Exception as e:
                block.append(f"(tool failed: {e})")
        if include_research:
            try:
                flags = research_flags(symbol, date=date)
                if flags:
                    block.append(f"Research triggered: {', '.join(flags)}")
                    block.append(get_company_research(symbol, date=date))
            except Exception as e:
                block.append(f"(research failed: {e})")
        parts.append("\n".join(block))
    return "\n".join(parts)


print("Task 2 tools loaded.")
list_tools()


Task 2 tools loaded.
Tool                   Status         Description
----------------------------------------------------------------------
  get_news             OK             Get recent news headlines (with publish time) for specific stocks. Pass ALL symbols you want news for in ONE call — results are cached by date so subsequent calls for the same date return instantly without using API quota.
  get_price_history    OK             Recent daily OHLCV prices for a stock (yfinance)
  get_rsi              OK             RSI momentum indicator (>70 overbought, <30 oversold) via Alpha Vantage
  get_macd             OK             MACD trend indicator computed from daily closes (bullish/bearish crossover)
  get_global_quote     OK             Latest price quote with daily change via Alpha Vantage
  get_company_research OK             SEC EDGAR evidence: recent 8-K/10-Q/10-K filings and quarterly EPS/revenue vs last year
  get_market_context   OK             Overall market regime: SPY tr

### Test Task 2

In [18]:
# Test your implementations
print("Testing MCP Tool Functions...")
print("=" * 60)

for func_name, func in [("get_rsi", get_rsi), ("get_macd", get_macd),
                          ("get_price_history", get_price_history),
                          ("get_global_quote", get_global_quote)]:
    try:
        result = func("AAPL")
        print(f"{func_name}('AAPL'):")
        print(f"  {result[:200]}")
    except NotImplementedError:
        print(f"{func_name}: Not yet implemented")
    except Exception as e:
        print(f"{func_name}: Error - {e}")

print("" + "=" * 60)

# Test get_extra_context (aggregates all tool outputs)
try:
    extra = get_extra_context(["AAPL"])
    assert isinstance(extra, str), f"Expected str, got {type(extra)}"
    print(f"get_extra_context(['AAPL']):")
    print(f"  Length: {len(extra)} chars")
    print(f"  Preview: {extra[:300]}")
except NotImplementedError:
    print("get_extra_context: Not yet implemented")
except Exception as e:
    print(f"get_extra_context: Error - {e}")

# Test that get_extra_context handles multiple symbols
try:
    extra_multi = get_extra_context(["AAPL", "NVDA"])
    assert isinstance(extra_multi, str), f"Expected str, got {type(extra_multi)}"
    assert len(extra_multi) >= len(extra), "Multi-symbol result should be >= single-symbol"
    print(f"get_extra_context(['AAPL', 'NVDA']):")
    print(f"  Length: {len(extra_multi)} chars")
except NotImplementedError:
    pass
except Exception as e:
    print(f"get_extra_context multi-symbol: Error - {e}")

print("" + "=" * 60)
print("Task 2 testing complete.")


Testing MCP Tool Functions...
get_rsi('AAPL'):
  [R-AAPL] RSI(14) for AAPL: 55.4 (neutral) [2026-10-30, Alpha Vantage]
get_macd('AAPL'):
  [D-AAPL] MACD for AAPL [2026-11-02]: MACD=3.1141, Signal=3.8275, Hist=-0.7134 (bearish)
get_price_history('AAPL'):
  [P-AAPL] Price history for AAPL (5d, yfinance):
  2026-10-27: O=343.17 H=347.78 L=342.39 C=347.10 V=39984400
  2026-10-28: O=346.92 H=347.18 L=343.52 C=343.83 V=35714300
  2026-10-29: O=348.00 H=351.4
get_global_quote('AAPL'):
  [Q-AAPL] Quote AAPL: $344.63 | change: -4.28 (-1.2267%) | vol: 67,841,555 [2026-10-30, Alpha Vantage]
get_extra_context(['AAPL']):
  Length: 834 chars
  Preview: [M-SPY] 2026-11-02 | SPY 760.18, 20-day avg 769.74, 50-day avg 767.38, 5-day return -0.68% -> mixed
[M-VIX] 2026-11-02 | VIX 18.8 (normal)
Market regime: RISK-ON
=== Extra Context for AAPL ===
[P-AAPL] Price history for AAPL (5d, yfinance):
  2026-10-27: O=343.17 H=347.78 L=342.39 C=347.10 V=3998440
get_extra_context(['AAPL', 'NVDA']):
  Length: 1522 

### Test my research tools

SEC EDGAR evidence cards, market context, research trigger, and point-in-time (backtest date) behaviour.

In [19]:
# Test my research tools (SEC EDGAR evidence + market context + research trigger)
print("Testing research tools...")
print("=" * 60)
print(get_company_research("AAPL"))
print()
print(get_company_research("AAPL", date="2026-08-01"))   # backtest date: only filings before Aug 1
print()
print(get_company_research("SPY"))                        # ETF: no SEC filings
print()
print(get_market_context())
print()
print("Research flags AAPL:", research_flags("AAPL"))
print("Research flags NVDA:", research_flags("NVDA"))
print()
print("Point-in-time check (backtest date 2026-08-01):")
print(get_price_history("AAPL", date="2026-08-01"))
print(get_rsi("AAPL", date="2026-08-01"))
print("=" * 60)
print("Alpha Vantage calls used today:", _av_calls_used(), "of", AV_DAILY_BUDGET)


Testing research tools...
=== SEC research: AAPL (Apple Inc.) as of today ===
  No 8-K/10-Q/10-K filings in the last 60 days.
  [F-AAPL-EPS] 2026-07-31 | SEC 10-Q | Diluted EPS for quarter ending 2026-06-27: 2.02 vs 1.57 a year earlier (+28.7% YoY)
  [F-AAPL-REV] 2026-07-31 | SEC 10-Q | Revenue for quarter ending 2026-06-27: $109.42B vs $94.04B a year earlier (+16.4% YoY)

=== SEC research: AAPL (Apple Inc.) as of 2026-08-01 ===
  [S-AAPL-1] 2026-07-31 | SEC 10-Q | quarterly report
  [S-AAPL-2] 2026-07-30 | SEC 8-K | results of operations (earnings release)
  [F-AAPL-EPS] 2026-07-31 | SEC 10-Q | Diluted EPS for quarter ending 2026-06-27: 2.02 vs 1.57 a year earlier (+28.7% YoY)
  [F-AAPL-REV] 2026-07-31 | SEC 10-Q | Revenue for quarter ending 2026-06-27: $109.42B vs $94.04B a year earlier (+16.4% YoY)

=== SEC research: SPY (SPDR S&P 500 ETF TRUST) as of today ===
  No 8-K/10-Q/10-K filings in the last 60 days.

[M-SPY] 2026-11-02 | SPY 760.18, 20-day avg 769.74, 50-day avg 767.38, 5-d

---
## Task 3: [TODO] RAG Memory System

### Objective
Implement a memory system that allows the trading agent to **learn from past trades**.

Two components:
1. **`TradingMemory`** class - stores situation-reflection pairs, retrieves similar experiences using BM25
2. **`reflect_on_trade()`** function - uses LLM to generate a lesson learned after a trade

### How it works

```
Trade executed -> Wait N loops -> Compare equity -> LLM reflection -> Store in memory
                                                                           |
Next decision <- Retrieve similar past reflections via BM25 <--------------+
```

### BM25 Retrieval
[BM25](https://en.wikipedia.org/wiki/Okapi_BM25) is a keyword-based ranking function:
- Scores documents based on **term frequency** and **inverse document frequency**
- The `rank_bm25` library provides `BM25Okapi`

```python
from rank_bm25 import BM25Okapi

corpus = ["apple stock rises today", "tesla earnings beat expectations"]
tokenized = [doc.lower().split() for doc in corpus]
bm25 = BM25Okapi(tokenized)
scores = bm25.get_scores("apple earnings".lower().split())
# scores = [0.45, 0.32]  -- first doc scores higher for "apple"
```

### Grading Criteria
- [ ] TradingMemory correctly stores and loads memories; BM25 retrieval returns relevant past reflections; edge cases handled (empty memory, no matching results). *Evidence: Test Task 3, and the first Task 5 iterations (memory is still empty there).*
- [ ] reflect_on_trade generates meaningful short reflections: each names the stock and the action and says whether the trade gained or lost. *Evidence: Test Task 3.*


### 3a. TradingMemory Class

In [17]:
class TradingMemory:
    """
    BM25-based memory system for storing and retrieving trading reflections.
    Each memory entry: {"situation": str, "reflection": str, "timestamp": int}
    """

    def __init__(self, memory_file: str = "trading_memory.json"):
        """
        Initialize the memory system.

        TODO:
        - Store memory_file path as self.memory_file
        - Initialize self.memories as empty list
        - Load existing memories from file if it exists
          (use json.load, handle file-not-found and parse errors gracefully)
        """
        # TODO: Implement
        if memory_file is None:
            memory_file = os.path.join(CACHE_DIR, "trading_memory.json")

        self.memory_file = memory_file
        self.memories = []
#
        if os.path.exists(self.memory_file):
            try:
                with open(self.memory_file, "r", encoding="utf-8") as f:
                    self.memories = json.load(f)
                print(f"Loaded {len(self.memories)} trading memories")
            except Exception as e:
                print(f"Memory file corrupted, starting fresh: {e}")
                self.memories = []

    def _save(self):
        """Persist memories to disk as JSON."""
        with open(self.memory_file, "w", encoding="utf-8") as f:
            json.dump(self.memories, f, ensure_ascii=False, indent=2)

    def add_memory(self, situation: str, reflection: str):
        """
        Store a new situation-reflection pair.

        TODO:
        - Append {"situation": situation, "reflection": reflection,
                   "timestamp": int(time.time())} to self.memories
        - Call self._save()
        """
        # TODO: Implement
        self.memories.append({
            "situation": situation,
            "reflection": reflection,
            "timestamp": int(time.time())
        })

        self._save()

    def get_memories(self, query: str, n_matches: int = 2) -> list:
        """
        Retrieve the most similar past reflections using BM25.

        Returns a list of reflection strings (best match first). Only results
        with a positive score are returned; empty memory or empty query -> [].
        Small-memory fix: with very few memories BM25's IDF can be zero or
        negative even for a matching word, so word overlap is used as a
        tie-breaking backup score.
        """
        if not self.memories or not query or not query.strip():
            return []
        def tokenize(t):   # words like "nvda", "risk-off"; skip numbers and 1-letter tokens
            return [w for w in re.findall(r"[a-z0-9\.\-]+", str(t).lower())
                    if len(w) > 1 and not w.replace(".", "").replace("-", "").isdigit()]
        corpus = [tokenize(m["situation"]) for m in self.memories]
        q = tokenize(query)
        if not q:
            return []
        bm25 = BM25Okapi(corpus)
        scores = bm25.get_scores(q)
        ranked = []
        for i, doc in enumerate(corpus):
            overlap = len(set(q) & set(doc))
            score = scores[i] if scores[i] > 0 else (overlap * 1e-3 if overlap else 0.0)
            if score > 0:
                ranked.append((score, i))
        ranked.sort(reverse=True)
        return [self.memories[i]["reflection"] for _, i in ranked[:n_matches]]


### 3b. Trade Reflection Function

In [18]:
def reflect_on_trade(tok, model, situation: str, decision: dict,
                     prev_equity: float, curr_equity: float) -> str:
    """
    Short reflection on a trade outcome (max 300 characters).

    The first part is written by Python so it always names the stock, the action
    and whether the trade gained or lost. The model then adds the lesson:
    what it expected, what happened, and whether the cited evidence helped.
    """
    try:
        pnl = curr_equity - prev_equity
        pnl_pct = pnl / prev_equity * 100 if prev_equity else 0.0
        action = str(decision.get("action", "?")).upper()
        symbol = decision.get("symbol", "?")
        qty = decision.get("qty", "?")
        outcome = "gained" if pnl >= 0 else "lost"
        head = f"{action} {qty} {symbol}: {outcome} ${abs(pnl):,.2f} ({pnl_pct:+.2f}%)."

        messages = [
            {"role": "system", "content": "You are a trading reflection assistant. Write ONE short "
             "sentence (max 25 words): a lesson for future trades. Mention whether the evidence "
             "used was helpful or misleading. No preamble."},
            {"role": "user", "content":
             f"Trade: {json.dumps({k: decision.get(k) for k in ('action', 'symbol', 'qty', 'reason', 'evidence')})}\n"
             f"Expected (the reason given): {decision.get('reason', '')}\n"
             f"What happened: the trade {outcome} ${abs(pnl):,.2f} ({pnl_pct:+.2f}%).\n"
             f"Situation: {situation[:400]}\nLesson:"}]
        text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = {k: v.to(model.device) for k, v in tok(text, return_tensors="pt").items()}
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=60, do_sample=False,
                                 pad_token_id=tok.eos_token_id)
        lesson = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        lesson = " ".join(lesson.split())
        return f"{head} Lesson: {lesson}"[:300]
    except Exception as e:
        print(f"  [Reflection] failed: {e}")
        return "No reflection generated."


### Test Task 3

In [22]:
# Test TradingMemory
try:
    test_mem = TradingMemory("test_memory.json")

    test_mem.add_memory(
        situation="AAPL bullish news, strong earnings, bought 10 shares",
        reflection="Buying on strong earnings news was profitable."
    )
    test_mem.add_memory(
        situation="TSLA bearish sentiment, missed earnings, sold 5 shares",
        reflection="Selling before bearish news helps protect capital."
    )
    test_mem.add_memory(
        situation="MSFT neutral news, market sideways, held position",
        reflection="Holding during uncertain times avoided transaction costs."
    )

    results = test_mem.get_memories("AAPL earnings report bullish", n_matches=2)
    print("Memory retrieval test:")
    print(f"  Query: 'AAPL earnings report bullish'")
    print(f"  Retrieved {len(results)} memories:")
    for i, r in enumerate(results, 1):
        print(f"    {i}. {r}")

    if os.path.exists("test_memory.json"):
        os.remove("test_memory.json")
    print("\nTradingMemory works!")
except NotImplementedError:
    print("Task 3a not yet implemented")
except Exception as e:
    print(f"Error: {e}")
    import traceback; traceback.print_exc()

# Test reflect_on_trade
try:
    reflection = reflect_on_trade(
        tok, model,
        situation="AAPL bullish news, bought 5 shares at $280",
        decision={"action": "buy", "symbol": "AAPL", "qty": 5},
        prev_equity=100000.0, curr_equity=100150.0
    )
    print(f"\nReflection test: {reflection}")
    print("reflect_on_trade works!")
except NotImplementedError:
    print("\nTask 3b (reflect_on_trade) not yet implemented")
except Exception as e:
    print(f"\nError: {e}")

Memory retrieval test:
  Query: 'AAPL earnings report bullish'
  Retrieved 2 memories:
    1. Buying on strong earnings news was profitable.
    2. Selling before bearish news helps protect capital.

TradingMemory works!

Reflection test: BUY 5 AAPL: gained $150.00 (+0.15%). Lesson: The evidence of buying at $280 and selling at $390 is misleading; the actual gain should have been higher than expected.
reflect_on_trade works!


---
## Task 4: [TODO] SFT Fine-tuning with LoRA

### Objective
Fine-tune the Qwen 1.5B model on **backtested trading data** using LoRA (Low-Rank Adaptation), then compare performance.

### Flow

```
Step 1: Backtest with BASE model
         Simulate ~60 trading days on historical prices
         LLM makes decisions each day (no real money)
         All (prompt, decisions, equity_change) logged automatically
                         │
Step 2: Auto-label data
         equity went UP   → label "good"
         equity went DOWN → label "bad"
                         │
Step 3: SFT fine-tune (YOUR IMPLEMENTATION)
         Filter "good" examples
         Tokenize as (prompt, completion) pairs
         LoRA: only train <0.1% of parameters
         Save adapter to Drive
                         │
Step 4: Backtest with FINE-TUNED model
         Same historical data, same starting cash
         Compare returns: base vs fine-tuned
```

### What you need to do

1. **`prepare_sft_dataset()`** - Filter labeled data, tokenize into training format
2. **`get_lora_model()`** - Configure LoRA adapters on the base model
3. **`train_sft()`** - Run the training loop
4. Compare backtest results

### Key Concepts

**LoRA** adds small trainable matrices to attention layers while freezing the base model:
- `r=8`: rank of the low-rank matrices (higher = more capacity but slower)
- `lora_alpha=16`: scaling factor (typically 2x r)
- `target_modules=["q_proj", "v_proj"]`: which layers to adapt
- Only ~0.07% of parameters are trainable → fast training on Colab

**SFT (Supervised Fine-Tuning)**: Train the model to reproduce "good" trading decisions.
The loss is only computed on the **completion** (decisions), not the prompt.

### Grading Criteria
- [ ] `prepare_sft_dataset` correctly filters and tokenizes data; LoRA config is reasonable (r, alpha, target modules); the training loop runs without errors and every epoch's loss is a normal number. *Evidence: 4c output: the dataset size matches the GOOD count (explain any difference in your report), the trainable-parameter line, the loss per epoch.*
- [ ] 4a and 4c are provided cells: run them, then write your observations and explanations in the training-strategy part of your report. The note at the top of 4a and of 4c says what to cover.

### 4a. Run Backtest & Collect Training Data

Run a backtest with the **base model** on historical data. This collects trading decisions and outcomes
without spending any paper money. Data is auto-labeled based on P&L.

> **Provided cell: run it, then explain what you see in your report.** It replays your agent with the base model over the last ~60 trading days (`run_backtest`, 1.14); the collector (1.13) logs the valid days, and `auto_label` marks each one GOOD or BAD from the next day's profit or loss. In your report: how many days were logged, the GOOD/BAD split, whether 1–2 sample labels make sense (the inspect cell below helps), and why you changed the labeling rule if you did.

In [24]:
# =============================================================================
# Run backtest with BASE model to collect training data
# =============================================================================

# Initialize a fresh collector (clears previous data)
import os as _os
_sft_path = _os.path.join(CACHE_DIR, "sft_data.jsonl")
if _os.path.exists(_sft_path):
    _os.remove(_sft_path)
    print("Cleared previous training data")

collector = TradingDataCollector()

# Run backtest - this takes a few minutes
print("Running backtest with BASE model...")
print("(Each day = 1 LLM inference, ~15-30 sec on T4)\n")
# [CHANGE] train on 40 older days; the most recent 20 days are held out for the 4c comparison
base_results = run_backtest(tok, model, collector=collector, n_days=40, skip_last=20)

# Auto-label: good if equity increased, bad if decreased
# [CHANGE] my labelling rule: forward return over 3 days with a 0.5% no-trade band
# (only uses prices up to the last training day, so the test period never leaks in)
collector.label_by_forward_return(fetch_historical_prices(), horizon=3, band=0.005,
                                  max_date=base_results["dates"][-1])

print("\n--- Training Data Summary ---")
collector.summary()

# Review labeled entries
print("\nRecent entries:")
collector.review(n=10)

# Manual override example (uncomment to use):
# collector.set_label(0, "good")   # override entry 0
# collector.set_label(3, "bad")    # override entry 3

print(f"\nGood examples for training: {len(collector.get_labeled('good'))}")
print(f"Corrected examples (wrong trades replaced by HOLD): {len(collector.get_labeled('corrected'))}")
print(f"Bad examples (not used in SFT): {len(collector.get_labeled('bad'))}")
print(f"Neutral / unlabelled: {len(collector.get_labeled('neutral'))} / {sum(1 for e in collector.entries if e.get('label') is None)}")

Running backtest with BASE model...
(Each day = 1 LLM inference, ~15-30 sec on T4)

Cached 127 trading days to Drive

BACKTEST  2026-08-10  →  2026-10-05  (40 days)
Initial cash: $100,000  |  Universe: 100 stocks
Agent: up to 6 tool-call turns per day
2026-08-10:
  [Agent turn 1] -> get_company_research({'symbol': 'JPM'})
    === SEC research: JPM (JPMORGAN CHASE & CO) as of 2026-08-10 ===
      [S-JPM-1] 2026-08-06 | SEC 10-Q | quarterly report
      [S-JPM-2] 2026-07-23 | SEC 8-K | other material event
      [S-JPM-3] 2026-07-23 | S
  [Agent turn 2] -> get_market_context({})
    [M-SPY] 2026-08-10 | SPY 773.03, 20-day avg 751.37, 50-day avg 747.56, 5-day return +2.03% -> uptrend
    [M-VIX] 2026-08-10 | VIX 15.5 (normal)
    Market regime: RISK-ON
  [Agent] Finalising - 3 turn(s), 2 tool call(s)
    Raw output: [
  {
    "action": "buy",
    "symbol": "AMD",
    "qty": 5,
    "evidence": ["R-AMD"],
    "reason": "strong growth and positive Q4 outlook"
  },
  {
    "action": "sell",
 

### 4a check: sample labels

Two logged days with their decisions and what happened next (for the report).

In [25]:
for e in [x for x in collector.entries if x.get("label")][:2]:
    print(f"{e.get('date')} | label={e['label']} | right={e.get('fwd_right')} wrong={e.get('fwd_wrong')}")
    for d in json.loads(e["completion"]):
        if d.get("action") != "hold":
            print(f"   {d['action']} {d['qty']} {d['symbol']} evidence={d.get('evidence')} | {d.get('reason')}")
    if e.get("corrected_completion"):
        print("   corrected ->", e["corrected_completion"][:300])
    print()


2026-08-14 | label=bad | right=0 wrong=1
   buy 5 MSFT evidence=['P-MSFT'] | Buy when risk-off

2026-08-17 | label=neutral | right=0 wrong=0
   sell 5 MSFT evidence=['P-MSFT'] | 



### Optional: Inspect & Edit SFT Training Data

After the backtest, `sft_data.jsonl` contains all logged (prompt, completion, label) pairs.
Use the cells below to review entries, remove low-quality ones, or upload a hand-edited version
before running `train_sft()`.


In [26]:
# =============================================================================
# OPTIONAL: Inspect & Edit sft_data.jsonl
# Run each sub-block independently as needed.
# =============================================================================

import os, json
# [LOCAL CHANGE] google.colab.files is Colab-only; files are already in stock_agent_cache.
# from google.colab import files

sft_path = os.path.join(CACHE_DIR, "sft_data.jsonl")

# -- 1. View all SFT entries -----------------------------------------------
if os.path.exists(sft_path):
    with open(sft_path, encoding="utf-8") as f:
        entries = [json.loads(l) for l in f if l.strip()]
    print(f"Found {len(entries)} SFT entries:")
    good = [e for e in entries if e.get('label') == 'good']
    bad  = [e for e in entries if e.get('label') == 'bad']
    print(f"  good: {len(good)}  |  bad: {len(bad)}\n")
    for i, e in enumerate(entries):
        label  = e.get('label', '?')
        pnl    = e.get('pnl', '?')
        compl  = e.get('completion', '')[:100].replace('\n', ' ')
        print(f"[{i:2d}] label={label}  pnl={pnl}")
        print(f"      completion: {compl}")
        print()
else:
    print("No sft_data.jsonl found yet. Run the backtest first.")

# -- 2. Download for local editing ----------------------------------------
# Uncomment to download:
# files.download(sft_path)

# -- 3. Upload an edited version back -------------------------------------
# Uncomment after editing locally:
# uploaded = files.upload()   # select your edited sft_data.jsonl
# for fname, content in uploaded.items():
#     with open(sft_path, 'wb') as f:
#         f.write(content)
# with open(sft_path, encoding="utf-8") as f:
#     entries = [json.loads(l) for l in f if l.strip()]
# print(f"Reloaded {len(entries)} SFT entries from uploaded file.")

# -- 4. Remove bad entries (keep only good for training) ------------------
# Uncomment to filter:
# if os.path.exists(sft_path):
#     with open(sft_path, encoding="utf-8") as f:
#         entries = [json.loads(l) for l in f if l.strip()]
#     good_only = [e for e in entries if e.get('label') == 'good']
#     with open(sft_path, 'w', encoding='utf-8') as f:
#         for e in good_only:
#             f.write(json.dumps(e) + '\n')
#     print(f"Kept {len(good_only)} good entries; removed {len(entries)-len(good_only)} bad.")


Found 25 SFT entries:
  good: 15  |  bad: 8

[ 0] label=bad  pnl=-77.53480000000854
      completion: [{"action": "buy", "symbol": "MSFT", "qty": 5, "evidence": ["P-MSFT"], "reason": "Buy when risk-off"

[ 1] label=neutral  pnl=-2.292250000013155
      completion: [{"action": "sell", "symbol": "MSFT", "qty": 5, "evidence": ["P-MSFT"], "reason": ""}]

[ 2] label=bad  pnl=-23.888850000003004
      completion: [{"action": "buy", "symbol": "NVDA", "qty": 10, "evidence": ["S-NVDA-1", "S-NVDA-2", "S-NVDA-3", "F-

[ 3] label=bad  pnl=-16.371899999983725
      completion: [{"action": "buy", "symbol": "NVDA", "qty": 10, "evidence": ["S-NVDA-1", "S-NVDA-2", "S-NVDA-3", "F-

[ 4] label=good  pnl=-22.38189999999304
      completion: [{"action": "sell", "symbol": "NVDA", "qty": 10, "evidence": ["P-NVDA", "R-NVDA", "D-NVDA", "S-NVDA-

[ 5] label=good  pnl=89.09280000001309
      completion: [{"action": "buy", "symbol": "NVDA", "qty": 10, "evidence": ["S-NVDA-1", "S-NVDA-2", "S-NVDA-3", "F-

[ 6] lab

### 4b. Implement SFT Fine-tuning

In [17]:
# =============================================================================
# 4b. SFT Fine-tuning with LoRA
# =============================================================================
import re


_ASSIST_RE = re.compile(r"(<\|im_start\|>assistant\n)(.*?<\|im_end\|>)", re.DOTALL)


def _tokenize_conversation(tok, prompt):
    """
    Tokenise the chat prompt and label the assistant's earlier TOOL-CALL turns.
    [FIX] The first training run labelled only the final decision JSON, so the
    fine-tuned model forgot the <tool_call> format and stopped calling tools.
    Assistant turns that were not proper tool calls (e.g. answers that needed a
    correction) stay masked, so the model does not learn from them.
    """
    ids, labels, pos = [], [], 0
    for m in _ASSIST_RE.finditer(prompt):
        pre = tok(prompt[pos:m.start(2)], add_special_tokens=False)["input_ids"]
        content = m.group(2)
        c_ids = tok(content, add_special_tokens=False)["input_ids"]
        ids += pre
        labels += [-100] * len(pre)
        ids += c_ids
        labels += c_ids if "<tool_call>" in content else [-100] * len(c_ids)
        pos = m.end(2)
    rest = tok(prompt[pos:], add_special_tokens=False)["input_ids"]
    return ids + rest, labels + [-100] * len(rest)


def prepare_sft_dataset(collector, tok, max_length=2048):
    """
    Turn "good" and "corrected" days into a tokenised SFT dataset.

    * text = prompt (the exact chat conversation the agent saw) + completion
      (the decision JSON) + <|im_end|> so the model learns to stop.
    * Labels: the final decision JSON AND the earlier tool-call turns;
      system/user/tool text gets -100 (no loss).
    * Too long? Cut from the LEFT (oldest text), so the completion is never
      cut off. If it were, every label would be -100 and the loss NaN.
    * No padding: batch_size is 1, so every example keeps its own length.
    """
    good = collector.get_labeled("good")
    corrected = collector.get_labeled("corrected")
    if not good and not corrected:
        raise ValueError("No 'good' labeled data found")

    end = "<|im_end|>"
    rows = {"input_ids": [], "attention_mask": [], "labels": []}
    n_tool_turns = 0
    for e in good + corrected:
        completion = e.get("corrected_completion") or e["completion"]
        p_ids, p_labels = _tokenize_conversation(tok, e["prompt"])
        n_tool_turns += e["prompt"].count("<tool_call>")
        c_ids = tok(completion + end, add_special_tokens=False)["input_ids"]
        if len(c_ids) >= max_length:
            continue
        ids = (p_ids + c_ids)[-max_length:]
        labels = (p_labels + c_ids)[-max_length:]
        rows["input_ids"].append(ids)
        rows["attention_mask"].append([1] * len(ids))
        rows["labels"].append(labels)

    print(f"SFT dataset: {len(good)} good + {len(corrected)} corrected = {len(rows['input_ids'])} examples "
          f"(avg length {sum(map(len, rows['input_ids'])) // max(1, len(rows['input_ids']))} tokens, "
          f"{n_tool_turns} tool-call turns also trained)")
    return Dataset.from_dict(rows)


def get_lora_model(base_model):
    """LoRA adapters on the attention query/value projections."""
    config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )
    # Saves GPU memory (8 GB laptop GPU) at a small speed cost
    base_model.gradient_checkpointing_enable()
    base_model.enable_input_require_grads()
    base_model.config.use_cache = False
    peft_model = get_peft_model(base_model, config)
    peft_model.print_trainable_parameters()
    return peft_model


def train_sft(peft_model, dataset, tok, epochs=2, lr=1e-4, batch_size=1):
    """
    Plain training loop; loss only on the assistant's tokens.
    [FIX] epochs 3 -> 2 and lr 2e-4 -> 1e-4: the first run (loss 0.20 on 15
    examples) overfitted and copied the training answers.
    """
    params = [p for p in peft_model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=lr)
    peft_model.train()
    for epoch in range(epochs):
        total_loss, steps, skipped = 0.0, 0, 0
        for i in range(0, len(dataset), batch_size):
            batch = dataset[i:i + batch_size]
            input_ids = torch.tensor(batch["input_ids"]).to(peft_model.device)
            attention_mask = torch.tensor(batch["attention_mask"]).to(peft_model.device)
            labels = torch.tensor(batch["labels"]).to(peft_model.device)

            outputs = peft_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            if not torch.isfinite(loss):          # never let one bad step break training
                optimizer.zero_grad()
                skipped += 1
                continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            optimizer.step()
            optimizer.zero_grad()
            total_loss += loss.item()
            steps += 1

        avg_loss = total_loss / max(1, steps)
        print(f"Epoch {epoch + 1}/{epochs} | Avg Loss: {avg_loss:.4f}"
              + (f" | skipped {skipped} non-finite steps" if skipped else ""))

    save_dir = os.path.join(CACHE_DIR, "lora_adapter")
    peft_model.save_pretrained(save_dir)
    print(f"LoRA adapter saved to {save_dir}")

    peft_model.eval()
    try:
        peft_model.base_model.model.gradient_checkpointing_disable()
    except Exception:
        pass
    peft_model.config.use_cache = True


### 4c. Re-run Backtest & Compare

Run the same backtest with the **fine-tuned model** and compare returns against the base model.

> **Provided cell: run it, then explain what you see in your report.** It trains LoRA with your 4b functions, then backtests the base model and the fine-tuned model on the last 20 trading days and prints the return and number of trades for each. In your report: how the two models differ (return, trades, whether they make real decisions each day), what you think caused it, and which conclusions you would not trust, and why.

In [18]:
# =============================================================================
# Compare base vs fine-tuned model via backtest
# =============================================================================
print("=" * 60)
print("SFT Fine-tuning Pipeline")
print("=" * 60)

collector = TradingDataCollector()
good_count = len(collector.get_labeled("good")) + len(collector.get_labeled("corrected"))  # [CHANGE] corrected days are training data too

if good_count == 0:
    print("\nNo 'good' labeled data found!")
    print("Run cell 4a first to backtest and collect training data.")
else:
    print(f"\nTraining on {good_count} 'good' examples...")

    try:
        # Step 1: Prepare dataset
        dataset = prepare_sft_dataset(collector, tok)
        print(f"Dataset ready: {len(dataset)} examples")

        # Step 2: Create LoRA model
        lora_model = get_lora_model(model)

        # Step 3: Train
        train_sft(lora_model, dataset, tok)
        print("\nTraining complete!")

        # Step 4: Backtest base model (comparison baseline)
        # Comment this part if you only want to run Fine-tuned model
        print("" + "=" * 60)
        print("BACKTEST: Base Model (comparison baseline)")
        print("=" * 60)
        # get_peft_model() adds LoRA to `model` in place, so `model` is no longer
        # the base model here. disable_adapter() turns LoRA off for this baseline run.
        with lora_model.disable_adapter():
            base_results = run_backtest(tok, lora_model, collector=None, n_days=20)

        # Step 5: Backtest with fine-tuned model
        print("" + "=" * 60)
        print("BACKTEST: Fine-tuned Model")
        print("=" * 60)
        ft_results = run_backtest(tok, lora_model, collector=None, n_days=20)

        # Step 6: Compare
        print("\n" + "=" * 60)
        print("COMPARISON: Base Model vs Fine-tuned Model")
        print("=" * 60)
        print(f"  Base model return:       {base_results['total_return']:>+8.2f}%")
        print(f"  Fine-tuned model return: {ft_results['total_return']:>+8.2f}%")
        diff = ft_results['total_return'] - base_results['total_return']
        print(f"  Improvement:             {diff:>+8.2f}%")
        print(f"")
        print(f"  Base trades:             {len(base_results['trades']):>8d}")
        print(f"  Fine-tuned trades:       {len(ft_results['trades']):>8d}")
        print(f"  Days with trades (base / fine-tuned): {base_results['trade_days']} / {ft_results['trade_days']} of {len(ft_results['dates'])}")
        print(f"  Buy-and-hold SPY:        {ft_results['spy_return']:>+8.2f}%")
        print(f"  Equal-weight focus list: {ft_results['ew_return']:>+8.2f}%")
        print(f"  Base agent stats:        {base_results['agent_stats']}")
        print(f"  Fine-tuned agent stats:  {ft_results['agent_stats']}")
        print("=" * 60)

        if diff > 0:
            print("\nFine-tuned model outperformed the base model!")
        elif diff < 0:
            print("\nBase model performed better. Consider:")
            print("  - Collecting more training data (increase n_days)")
            print("  - Adjusting labeling threshold")
            print("  - Tuning LoRA hyperparameters (r, alpha, lr)")
        else:
            print("\nBoth models performed equally.")

    except NotImplementedError as e:
        print(f"\nNot yet implemented: {e}")
    except Exception as e:
        print(f"\nError: {e}")
        import traceback; traceback.print_exc()

SFT Fine-tuning Pipeline
Loaded 25 existing training entries

Training on 15 'good' examples...
SFT dataset: 15 good + 0 corrected = 15 examples (avg length 1994 tokens, 60 tool-call turns also trained)
Dataset ready: 15 examples
trainable params: 1,089,536 || all params: 1,544,803,840 || trainable%: 0.0705
Epoch 1/2 | Avg Loss: 0.5978
Epoch 2/2 | Avg Loss: 0.5184


LoRA adapter saved to C:\Users\syeda\trading_agent\stock_agent_cache\lora_adapter

Training complete!
BACKTEST: Base Model (comparison baseline)
Loaded cached price data (127 trading days)

BACKTEST  2026-10-06  →  2026-11-02  (20 days)
Initial cash: $100,000  |  Universe: 100 stocks
Agent: up to 6 tool-call turns per day
2026-10-06:
  [Agent turn 1] -> get_company_research({'symbol': 'META'})
    === SEC research: META (Meta Platforms, Inc.) as of 2026-10-06 ===
      No 8-K/10-Q/10-K filings in the last 60 days.
      [F-META-EPS] 2026-07-30 | SEC 10-Q | Diluted EPS for quarter ending 2026-06-30: 6.1
  [Agent] Finalising - 2 turn(s), 1 tool call(s)
    Raw output: [{"action": "hold", "symbol": "META", "reason": "No new information available."}]
  [Agent] Finalising - 3 turn(s), 1 tool call(s)
    Raw output: [
  {
    "action": "sell",
    "symbol": "JPM",
    "qty": 2,
    "evidence": ["M-JPM"]
  },
  {
    "action": "buy",
    "symbol": "NVDA",
    "qty": 5,
    "evidence": ["R-NVD

### My improvement: before/after

Research + evidence verification switched off (before) vs on (after), base model, same 20 held-out days. Run after 4c.

In [19]:
# =============================================================================
# My improvement — before/after: research + evidence verification OFF vs ON
# "After" (ON) = the base-model run from 4c above. "Before" (OFF) = same base
# model, same 20 held-out days, same seeds, but no SEC research tool, no
# research trigger and no evidence check.
# =============================================================================
USE_RESEARCH, USE_VERIFICATION = False, False
try:
    with lora_model.disable_adapter():
        before = run_backtest(tok, lora_model, collector=None, n_days=20)
finally:
    USE_RESEARCH, USE_VERIFICATION = True, True     # always switch back on

after = base_results
print("\n" + "=" * 60)
print("IMPROVEMENT: research + verification OFF (before) vs ON (after)")
print("=" * 60)
print(f"  Return:              {before['total_return']:+.2f}%  ->  {after['total_return']:+.2f}%")
print(f"  Trades:              {len(before['trades'])}  ->  {len(after['trades'])}")
print(f"  Days with trades:    {before['trade_days']}  ->  {after['trade_days']}  (of {len(after['dates'])})")
print(f"  Blocked unsupported: {before['agent_stats']['blocked_unsupported']}  ->  {after['agent_stats']['blocked_unsupported']}")
print(f"  Buy-and-hold SPY:    {after['spy_return']:+.2f}%")


Loaded cached price data (127 trading days)

BACKTEST  2026-10-06  →  2026-11-02  (20 days)
Initial cash: $100,000  |  Universe: 100 stocks
Agent: up to 6 tool-call turns per day
2026-10-06:
  [Agent turn 1] -> get_price_history({'symbol': 'AAPL', 'period': '5d'})
    [P-AAPL] Price history for AAPL (5d, yfinance):
      2026-09-30: O=330.80 H=339.50 L=330.14 C=333.02 V=49988600
      2026-10-01: O=330.00 H=332.48 L=325.81 C=330.32 V=36306300
      2026-10-02: O=333.26 H=334.5
  [Agent turn 1] -> get_price_history({'symbol': 'NVDA', 'period': '5d'})
    [P-NVDA] Price history for NVDA (5d, yfinance):
      2026-09-30: O=229.27 H=232.37 L=228.17 C=228.38 V=121732200
      2026-10-01: O=229.98 H=232.29 L=228.16 C=230.86 V=98591400
      2026-10-02: O=236.06 H=237.
  [Agent turn 2] -> get_rsi({'symbol': 'AAPL', 'time_period': 14})
    [R-AAPL] RSI(14) for AAPL: 54.6 (neutral) [2026-10-06, Alpha Vantage]
  [Agent turn 2] -> get_rsi({'symbol': 'NVDA', 'time_period': 14})
    [R-NVDA] RSI(14

---
## Task 5: [TODO] Full Trading Loop Integration

### Objective
Wire Tasks 2-3 together with the pre-written infrastructure into a complete trading loop, using the **fine-tuned model** from Task 4.

Each iteration:
1. Get current account state
2. Fetch financial news (from Drive cache)
3. Fetch extra context from your MCP tool functions (Task 2)
4. Check pending trades for reflection (after delay)
5. Retrieve similar past experiences from memory
6. Make LLM-powered decisions
7. Execute orders
8. Queue trades for future reflection

### Important Notes
- The loop runs for `n_iterations` (not infinite)
- Use `time.sleep(LOOP_SECONDS)` between iterations
- `pending_reflections` tracks trades awaiting reflection
- A trade is ready for reflection when `loop_count - trade["loop"] >= REFLECTION_DELAY_LOOPS`

### Grading Criteria
- [ ] All steps are correctly wired together; error handling prevents one failed step from crashing the loop; `extra_context` from Task 2 is passed to `llm_agent_decide()`; memory from Task 3 is used for reflections and retrieval; orders are only submitted for buy/sell decisions (not hold). *Evidence: at least 10 consecutive iterations, each showing the memory retrieval and the Task 2 data, and at least one order with its status.*
- [ ] Live week (11/9–11/13): the output of every trading day, plus the Alpaca order history and final balance. *Evidence: the daily logs in `trading_logs/` on Drive; orders and the final balance are checked in your Alpaca account.*

In [19]:
def run_trading_loop(tc, tok, model, memory=None, collector=None,
                     n_iterations=5, max_minutes=None):
    """
    Live paper-trading loop. Each iteration:
      0. market clock        1. account state (+ daily loss limit)
      2. news                3. Task 2 tool data (extra_context)
      4. reflections on trades older than REFLECTION_DELAY_LOOPS
      4.5 collector outcomes 5. memory retrieval
      6. agent decision      6.5 log for SFT
      7. orders (buy/sell only, only while the market is open)
    Every step has its own try/except so one failure never stops the loop.
    """
    pending_reflections = []
    pending_outcomes = []
    start_time = time.time()
    start_equity = None

    for loop_count in range(1, n_iterations + 1):
        now_et = datetime.datetime.now(datetime.timezone.utc) - datetime.timedelta(hours=5)
        print(f"\n{'=' * 60}")
        print(f"LOOP #{loop_count} / {n_iterations} | {now_et:%Y-%m-%d %H:%M} ET (approx) | "
              f"Pending reflections: {len(pending_reflections)}")
        print('=' * 60)

        # ---- Step 0: is the market open? ----
        try:
            is_open = bool(tc.get_clock().is_open)
        except Exception as e:
            print(f"  [Clock] could not check market clock: {e}")
            is_open = True
        print(f"  [Market] {'OPEN' if is_open else 'CLOSED - decisions are made but no orders are sent'}")

        # ---- Step 1: account state ----
        try:
            state = get_state(tc)
        except Exception as e:
            print(f"  [State] failed: {e} - retrying next loop")
            time.sleep(LOOP_SECONDS)
            continue
        if start_equity is None:
            start_equity = state["equity"]
        change = state["equity"] / start_equity - 1
        state["block_buys"] = change <= -DAILY_LOSS_LIMIT
        print(f"  [Account] equity ${state['equity']:,.2f} | cash ${state['cash']:,.2f} | "
              f"since start {change:+.2%}" + ("  -> LOSS LIMIT: no new buys" if state["block_buys"] else ""))
        for p in state["positions"]:
            print(f"    {p['symbol']}: {p['qty']:g} sh, value ${p['market_value']:,.2f}")

        # stocks to look at closely: current positions + the 3 biggest movers today
        held = [p["symbol"] for p in state["positions"] if p["symbol"] in WHITELIST]
        movers = []
        try:
            moves = []
            for s in FOCUS_LIST:
                c = _history_until(s, None)["Close"]
                moves.append((abs(c.iloc[-1] / c.iloc[-2] - 1), s))
            movers = [s for _, s in sorted(moves, reverse=True)[:3]]
        except Exception as e:
            print(f"  [Movers] failed: {e}")
        symbols = list(dict.fromkeys(held + movers))[:5] or FOCUS_LIST[:3]
        print(f"  [Focus] {symbols}")

        # ---- Step 2: news ----
        try:
            news_context = get_news_for_symbols(symbols)
        except Exception as e:
            news_context = ""
            print(f"  [News] failed: {e}")

        # ---- Step 3: Task 2 tool data ----
        try:
            extra_context = get_extra_context(symbols, include_research=USE_RESEARCH)
        except Exception as e:
            extra_context = ""
            print(f"  [Task 2 data] failed: {e}")
        print("  [Task 2 data] " + extra_context[:300].replace("\n", "\n    "))

        # ---- Step 4: reflections ----
        if memory is not None:
            for t in list(pending_reflections):
                if loop_count - t["loop"] < REFLECTION_DELAY_LOOPS:
                    continue
                try:
                    d = t["decision"]
                    now_price = _live_price(d["symbol"])
                    sign = 1 if d["action"] == "buy" else -1
                    trade_pnl = sign * (now_price - t["price"]) * d["qty"]
                    reflection = reflect_on_trade(tok, model, t["situation"], d,
                                                  t["equity"], t["equity"] + trade_pnl)
                    memory.add_memory(t["situation"], reflection)
                    print(f"  [Reflection] {reflection}")
                except Exception as e:
                    print(f"  [Reflection] failed: {e}")
                pending_reflections.remove(t)

        # ---- Step 4.5: collector outcomes ----
        for entry in list(pending_outcomes):
            if loop_count - entry["loop"] >= REFLECTION_DELAY_LOOPS:
                if collector is not None:
                    collector.update_outcome(entry["idx"], state["equity"])
                pending_outcomes.remove(entry)

        # ---- Step 5: memory retrieval ----
        regime = "risk-off" if "RISK-OFF" in extra_context else "risk-on"
        summary = f"{' '.join(symbols)} {regime} " + news_context[:300]
        past_reflections = []
        if memory is not None:
            try:
                past_reflections = memory.get_memories(summary, MEMORY_TOP_K)
            except Exception as e:
                print(f"  [Memory] retrieval failed: {e}")
        print(f"  [Memory] {len(memory.memories) if memory else 0} stored, "
              f"{len(past_reflections)} retrieved")
        for r in past_reflections:
            print(f"    - {r}")

        # ---- Step 6: decision ----
        try:
            context = (news_context[:1500] + "\n" + extra_context)[:3000]
            decisions, sft_prompt, sft_completion, tool_calls_made = llm_agent_decide(
                tok, model, state, universe=FOCUS_LIST, date=None,
                extra_context=context, past_reflections=past_reflections)
        except Exception as e:
            print(f"  [Agent] failed: {e}")
            time.sleep(LOOP_SECONDS)
            continue
        print("  [Decisions]")
        for d in decisions:
            print(f"    {d['action'].upper():4s} {d['symbol']:<5} qty={str(d.get('qty')):>4} "
                  f"evidence={d.get('evidence', [])} | {d.get('reason', '')[:80]}")

        # ---- Step 6.5: log for SFT ----
        if collector is not None:
            try:
                idx = collector.log(sft_prompt, sft_completion, decisions, state["equity"], tool_calls_made)
                if idx is not None:
                    collector.entries[idx]["date"] = datetime.date.today().isoformat()
                    pending_outcomes.append({"idx": idx, "loop": loop_count})
            except Exception as e:
                print(f"  [Collector] failed: {e}")

        # ---- Step 7: orders ----
        for d in decisions:
            if d["action"] not in ("buy", "sell"):
                continue
            if not is_open:
                print(f"  [ORDER] {d['action']} {d['qty']} {d['symbol']} NOT sent (market closed)")
                continue
            try:
                result = submit_order(tc, d)
                print(f"  [ORDER] {result}")
            except Exception as e:
                print(f"  [ORDER] {d['action']} {d['symbol']} failed: {e}")
                continue
            if memory is not None:
                try:
                    price = _live_price(d["symbol"])
                except Exception:
                    price = 0.0
                pending_reflections.append({
                    "loop": loop_count, "decision": d, "equity": state["equity"], "price": price,
                    "situation": f"{d['symbol']} {d['action']} {d['qty']} | {regime} | "
                                 f"reason: {d.get('reason', '')} | evidence: {' '.join(d.get('evidence', []))}"})

        # ---- stop conditions + wait ----
        minutes = (time.time() - start_time) / 60
        if max_minutes and minutes >= max_minutes:
            print(f"\nReached RUN_MINUTES ({max_minutes} min) - stopping.")
            break
        if loop_count < n_iterations:
            time.sleep(LOOP_SECONDS)


### Run the Trading Loop

Use the **fine-tuned model** (`lora_model`) from Task 4 for live paper trading.

#### Live week (11/9–11/13)

- Each trading day, run the loop during market hours (9:30–16:00 ET), ideally for 2–4 hours (longer if you can; set `LOOP_SECONDS` and `n_iterations` to match).
- **Do not rerun 4a or 4c on live days.** 4a clears your training data and 4c retrains the model, which would change your frozen model. Run Task 1 to Task 3 and the `run_trading_loop` cell, then the cell below: it loads the adapter Task 4 saved on Drive by itself.
- **The daily log is kept for you.** Everything the cell below prints is also saved to `CACHE_DIR/trading_logs/<date>.log` on Drive, so make your loop print each decision and order clearly. You submit these logs on 11/19, and your report reflects on the week.

In [20]:
# Initialize memory system
try:
    memory = TradingMemory(os.path.join(CACHE_DIR, "trading_memory.json"))  # keep on Drive
    print(f"Memory loaded: {len(memory.memories)} past entries")
except:
    memory = None
    print("Memory not available (implement Task 3 first)")

# Initialize data collector
collector = TradingDataCollector()

# Use fine-tuned model from Task 4. On live days (4a and 4c skipped), load the adapter Task 4 saved on Drive.
try:
    trading_model = lora_model
    model_note = "fine-tuned LoRA model from Task 4 (this session)"
    print("Using fine-tuned LoRA model from Task 4")
except NameError:
    try:
        from peft import PeftModel
        trading_model = PeftModel.from_pretrained(model, os.path.join(CACHE_DIR, "lora_adapter"))
        trading_model.eval()
        model_note = "fine-tuned LoRA adapter loaded from Drive"
        print("Loaded the fine-tuned LoRA adapter saved on Drive")
    except Exception as e:
        trading_model = model
        model_note = f"BASE model (no saved LoRA adapter: {type(e).__name__})"
        print(f"WARNING: no saved LoRA adapter ({type(e).__name__}), using the BASE model. Run Task 4 first.")

# Daily trading log: everything printed below also goes to a file on Drive.
import contextlib, sys
log_dir = os.path.join(CACHE_DIR, "trading_logs")
os.makedirs(log_dir, exist_ok=True)
now_utc = datetime.datetime.now(datetime.timezone.utc)
log_path = os.path.join(log_dir, f"{now_utc.date().isoformat()}.log")

class _Tee:
    """Writes to the notebook output and to the log file at the same time."""
    def __init__(self, *streams):
        self.streams = streams
    def write(self, text):
        for s in self.streams:
            s.write(text)
            s.flush()
        return len(text)
    def flush(self):
        for s in self.streams:
            s.flush()

with open(log_path, "a", encoding="utf-8") as log_file, contextlib.redirect_stdout(_Tee(sys.stdout, log_file)):
    print(f"\n===== Run started {now_utc.strftime('%Y-%m-%d %H:%M')} UTC =====")
    print(f"  Model: {model_note}")
    print("\nStarting trading loop...")
    print(f"  Symbols: {len(WHITELIST)} stocks")
    print(f"  Iterations: up to 80, max {RUN_MINUTES} minutes")
    print(f"  Interval: {LOOP_SECONDS}s")
    print("-" * 60)

    try:
        run_trading_loop(tc, tok, trading_model, memory=memory,
                         collector=collector, n_iterations=80,
                         max_minutes=RUN_MINUTES)   # [CHANGE] stop after RUN_MINUTES
        print("\n\nTrading loop completed successfully!")
        print("\nData collected:")
        collector.summary()
    except NotImplementedError:
        print("Task 5 not yet implemented")
    except Exception as e:
        print(f"\nError: {e}")
        import traceback; traceback.print_exc(file=sys.stdout)

print(f"\nTrading log saved to {log_path}")


Loaded 247 trading memories
Memory loaded: 247 past entries
Loaded 336 existing training entries
Loaded the fine-tuned LoRA adapter saved on Drive

===== Run started 2026-11-13 15:00 UTC =====
  Model: fine-tuned LoRA adapter loaded from Drive

Starting trading loop...
  Symbols: 100 stocks
  Iterations: up to 80, max 240 minutes
  Interval: 120s
------------------------------------------------------------

LOOP #1 / 80 | 2026-11-13 10:00 ET (approx) | Pending reflections: 0
  [Market] OPEN
  [Account] equity $98,448.00 | cash $84,835.38 | since start +0.00%
    NVDA: 84 sh, value $13,610.52
  [Focus] ['NVDA', 'TSLA', 'META']
  [Task 2 data] [M-SPY] 2026-11-13 | SPY 753.77, 20-day avg 758.99, 50-day avg 764.42, 5-day return +2.51% -> downtrend
    [M-VIX] 2026-11-13 | VIX 18.9 (normal)
    Market regime: RISK-OFF (prefer smaller positions)
    === Extra Context for NVDA ===
    [P-NVDA] Price history for NVDA (5d, yfinance):
      2026-11-09: O=173.07 H=17
  [Memory] 247 stored, 2 retr

$XOM: possibly delisted; no price data found  (period=1y)


  [Movers] failed: no price data for XOM
  [Focus] ['MSFT', 'NVDA']
  [Task 2 data] [M-SPY] 2026-11-13 | SPY 752.52, 20-day avg 758.93, 50-day avg 764.39, 5-day return +2.34% -> downtrend
    [M-VIX] 2026-11-13 | VIX 19.7 (normal)
    Market regime: RISK-OFF (prefer smaller positions)
    === Extra Context for MSFT ===
    [P-MSFT] Price history for MSFT (5d, yfinance):
      2026-11-09: O=544.49 H=54
  [Memory] 284 stored, 2 retrieved
    - SELL 10 NVDA: gained $19.70 (+0.02%). Lesson: The evidence of EPS and revenue growth was misleading; actual gain was much higher than expected profit taking.
    - BUY 4 NVDA: gained $2.36 (+0.00%). Lesson: The evidence supporting the decision to buy NVIDIA shares was helpful as it aligned with bullish sentiment based on strong Q2 earnings and revenue expectations.
  [Agent turn 1] -> get_company_research({'symbol': 'NVDA'})
    === SEC research: NVDA (NVIDIA CORP) as of today ===
      [S-NVDA-1] 2026-11-12 | SEC 8-K | director or officer change
 

### Optional: Inspect & Edit Memory

Use these cells to view, manually edit, download, or upload `trading_memory.json`.
You can add your own reflections or remove bad ones before the trading loop uses them.


In [ ]:
# =============================================================================
# OPTIONAL: Inspect & Edit trading_memory.json
# Run each sub-block independently as needed.
# =============================================================================

import os, json
# [LOCAL CHANGE] google.colab.files is Colab-only; files are already in stock_agent_cache.
# from google.colab import files

memory_path = os.path.join(CACHE_DIR, "trading_memory.json")

# -- 1. View all memory entries --------------------------------------------
if os.path.exists(memory_path):
    with open(memory_path, encoding="utf-8") as f:
        memories = json.load(f)
    print(f"Found {len(memories)} memory entries:\n")
    for i, m in enumerate(memories):
        label = m.get('label', '?')
        pnl   = m.get('pnl', '?')
        refl  = m.get('reflection', '')[:120].replace('\n', ' ')
        print(f"[{i:2d}] label={label}  pnl={pnl}")
        print(f"      {refl}")
        print()
else:
    print("No trading_memory.json found yet. Run the backtest first.")

# -- 2. Download for local editing ----------------------------------------
# Uncomment to download:
# files.download(memory_path)

# -- 3. Upload an edited version back -------------------------------------
# Uncomment the block below after you have edited the file locally:
# uploaded = files.upload()   # select your edited trading_memory.json
# for fname, content in uploaded.items():
#     with open(memory_path, 'wb') as f:
#         f.write(content)
# with open(memory_path, encoding="utf-8") as f:
#     memories = json.load(f)
# print(f"Reloaded {len(memories)} memories from uploaded file.")

# -- 4. Delete a bad entry by index ---------------------------------------
# Example: remove entry at index 2
# BAD_IDX = 2
# if os.path.exists(memory_path):
#     with open(memory_path, encoding="utf-8") as f:
#         memories = json.load(f)
#     removed = memories.pop(BAD_IDX)
#     with open(memory_path, 'w', encoding='utf-8') as f:
#         json.dump(memories, f, indent=2)
#     print(f"Removed entry {BAD_IDX}: {removed.get('reflection','')[:60]}")
#     print(f"{len(memories)} entries remain.")


---
## Optional Challenges

### 1. DPO (Direct Preference Optimization)
Instead of SFT (training only on "good" examples), use **DPO** to train on preference pairs:

```python
from trl import DPOTrainer, DPOConfig

# Format: (prompt, chosen_response, rejected_response)
# "chosen" = decisions labeled "good"
# "rejected" = decisions labeled "bad"

dpo_dataset = Dataset.from_dict({
    "prompt": [e["prompt"] for e in pairs],
    "chosen": [e["good_completion"] for e in pairs],
    "rejected": [e["bad_completion"] for e in pairs],
})

config = DPOConfig(
    output_dir="dpo_output",
    per_device_train_batch_size=1,
    num_train_epochs=3,
    learning_rate=5e-5,
)

trainer = DPOTrainer(
    model=lora_model,
    ref_model=None,        # with a LoRA model, trl uses it with the adapter off as the reference
    train_dataset=dpo_dataset,
    processing_class=tok,
    args=config,
)
trainer.train()
```

**Key difference**: SFT teaches "do this", DPO teaches "prefer this over that" — generally produces better-calibrated models.

### 2. Vector Database RAG
Replace BM25 with semantic search:
- Use sentence embeddings (`sentence-transformers`)
- Store in FAISS vector index
- Retrieve by cosine similarity

### 3. Advanced Trading Strategy
- Position sizing based on confidence scores
- Stop-loss and take-profit logic
- Portfolio performance tracking with matplotlib

### 4. More Data Sources
- SEC filings (EDGAR API)
- Social media sentiment (Reddit, Twitter)
- Options chain data
- Earnings calendar integration